# Prithvi-EO-2.0 burn-scar mapping — DIMER E2E segmentation fine-tuning tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/prithvi-burnscar-segmentation-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/prithvi-burnscar-segmentation-pipeline/blob/main/tutorials/prithvi_burnscar_segmentation_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-ibm--nasa--geospatial%2FPrithvi--EO--2.0--300M--BurnScars-ffcc4d?style=flat)](https://huggingface.co/ibm-nasa-geospatial/Prithvi-EO-2.0-300M-BurnScars) [![Upstream](https://img.shields.io/badge/Upstream-NASA--IMPACT%2FPrithvi--EO--2.0-181717?style=flat&logo=github&logoColor=white)](https://github.com/NASA-IMPACT/Prithvi-EO-2.0) [![Paper](https://img.shields.io/badge/arXiv-2412.02732-b31b1b.svg)](https://arxiv.org/abs/2412.02732)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** burn-scar segmentation of six-band HLS scenes with a Prithvi-EO-2.0 ViT-L encoder and U-Net decoder, held-out IoU/F1 against a not-burned baseline, and bounded fine-tuning of the decoder to labelled scenes

**This notebook is standalone.** It carries the repository's package (3 modules under `src/prithvi_burnscar_segmentation_pipeline/`, at revision `uncommitted`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `a3f2c410e45b8ac7417976614528a872f024d831` (~1317 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh **GPU** runtime installs the pinned dependencies (torch, torchvision, terratorch and its stack, tifffile, numpy, safetensors, huggingface-hub), stages and digest-verifies the pinned Prithvi burn-scar checkpoint (1.30 GB) from the Hub, statically audits the Lightning pickle against an allow-list, converts it once into safetensors with a pinned digest, rebuilds the architecture from the installed `terratorch` package and loads it strictly, fetches the digest-pinned HLS Burn Scars tarball (2.6 GB, no credential) and extracts exactly the 44 pinned scenes and masks, validates them and assigns the model repository's roles (24 training, 8 validation, 12 test), segments the held-out scenes with the frozen model and scores them against the not-burned baseline, runs a bounded fine-tuning of the neck, U-Net decoder and head, scores the same scenes again, segments the three example scenes shipped with the upstream repository, exports the adapter as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify prediction parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). On a T4 the whole path takes a few minutes of model time after the downloads; the terratorch install and the tarball are the slowest steps.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to supply your own labelled scenes as a zip holding `pairs.csv` (columns `id`, `image`, `label`) beside six-band 512 × 512 GeoTIFF chips (blue, green, red, narrow NIR, SWIR 1, SWIR 2 — surface reflectance in [0, 1] or × 10 000) and single-band label rasters (0 = not burned, 1 = burn scar, −1 = no data); at least four chips with some burn scar. Your chips are split by seed into training, validation and test sets and flow through the same contract — validation, frozen baseline, adaptation, held-out evaluation, inference, artifact export and reload parity. The expected schema, the ceilings and the privacy guidance are stated in the Prerequisites and in Section 4, and uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

Prithvi-EO-2.0 (Szwarcman et al., 2024) is NASA and IBM's foundation model for Harmonized Landsat Sentinel-2 imagery: a ViT-L masked autoencoder pretrained on 4.2 M global multispectral samples. The checkpoint packaged here is the upstream authors' fine-tune for burn-scar mapping — the 300 M-parameter encoder, a learned pyramid neck over four encoder depths, a U-Net decoder and a two-class head — trained on the 804 labelled HLS scenes of the HLS Burn Scars dataset (2018–2021, contiguous United States) with TerraTorch, on the non-overlapping splits the authors publish beside the checkpoint.

Two things about this row are handled in the open. **The upstream asset is a pickle** — a PyTorch Lightning checkpoint. Section 3 downloads and digest-verifies it, statically lists every global the pickle would import (a state dict of tensors and nothing else), refuses anything outside that allow-list, unpickles it exactly once through torch's weights-only loader, and writes a safetensors file whose digest is pinned in the carried module; the model you run is rebuilt from the installed `terratorch` package and loads that file strictly. **The dataset ships as one 2.6 GB tarball**, so Section 4 pins the tarball by size and digest, streams through it once and copies out exactly the 88 pinned members (each pinned again by size and digest, no `extractall`, no paths taken from the archive), and leaves the other 1,522 alone. **The model is already fine-tuned on this dataset**, so the bounded adaptation in Section 6 is a demonstration of the contract, selected by validation loss with the frozen model as epoch 0; the point of the contract is the same recipe applied to *your* labelled scenes.

**Learning objectives:** install the pinned runtime; inspect the carried pipeline, dataset and metrics modules; stage and digest-verify a pickled checkpoint, read its static audit and see it converted into safetensors; extract pinned members from a digest-verified tarball and validate real labelled multispectral scenes with an ignore class; read pixel IoU, F1, precision and recall against a not-burned baseline; run a bounded decoder fine-tuning with explicit hyperparameters and frozen BatchNorm statistics; compare the adapted and frozen models on the same held-out scenes; segment new scenes; and export a safetensors adapter that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** burn severity or dNBR estimation, active-fire detection, temporal stacks (the packaged fine-tune is single-date), tiling of scenes larger than 512 × 512, atmospheric correction, cloud masking, the published benchmark scores, and any claim that a 44-scene sample stands in for an operational evaluation. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported **GPU** runtime (Google Colab T4 or better, or a Jupyter kernel with a CUDA GPU and Python 3.12): the ViT-L encoder runs in float16 autocast and the default adaptation needs about 2.5 GB of GPU memory; on CPU one 512 × 512 scene takes tens of seconds and the adaptation would take an hour. About 6 GB of disk is needed for the checkpoint, its conversion and the tarball; the `terratorch` install pulls torchgeo, lightning and their dependencies and takes several minutes.
- **Knowledge:** what a multispectral surface-reflectance scene is (bands, scaling, no-data), what a pixel-wise segmentation mask and an ignore class are, and how IoU, precision and recall are read against a majority baseline.
- **Executable serialization handled explicitly:** the pinned checkpoint is a pickle. It is digest-verified, statically audited against an allow-list (audit digest pinned) and unpickled **once** through torch's weights-only loader to produce the safetensors the model is actually loaded from. No Hub-hosted Python module is imported; `terratorch` is installed from PyPI at a pinned version.
- **Data contract:** a record is `{{id, image, label}}` — a (6, 512, 512) reflectance array (or a GeoTIFF; 13-band Sentinel-2 L1C files are reduced to the six HLS-equivalent bands) with values in [0, 1] or × 10 000, no-data 0 or −9999, and a (512, 512) mask with 0 / 1 / −1. Validation is structural: nothing checks that the bands are the right six in the right order, that the reflectance is corrected, or that the label belongs to the scene.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there — commercial imagery under licence or unreleased fire-damage assessments are exactly that. The default path uploads nothing.
- **External access (data):** besides the model snapshot, the default path fetches one pinned object — the 2.6 GB `hls_burn_scars.tar.gz` of the Hugging Face dataset `ibm-nasa-geospatial/hls_burn_scars` at an immutable revision — over HTTPS, digest-verified before any member is read; the dataset is CC BY 4.0 (NASA IMPACT / University of Alabama in Huntsville).
- **External access:** the Hugging Face Hub only, to fetch the pinned `ibm-nasa-geospatial/Prithvi-EO-2.0-300M-BurnScars` snapshot (~1317 MB in total) at revision `a3f2c410e45b…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `timm`, `lightning`, `tifffile` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'terratorch==1.2.13',
    'tifffile==2026.9.15',
    'numpy==2.5.3',
    'safetensors==0.8.0',
    'huggingface-hub==1.32.0',
]
NOTEBOOK_SOURCE = {
    'repository': 'prithvi-burnscar-segmentation-pipeline',
    'repository_revision': 'uncommitted',
    'embedded_module': 'src/prithvi_burnscar_segmentation_pipeline/pipeline.py',
    'embedded_modules': ['src/prithvi_burnscar_segmentation_pipeline/pipeline.py', 'src/prithvi_burnscar_segmentation_pipeline/metrics.py', 'src/prithvi_burnscar_segmentation_pipeline/samples.py'],
    'module_sha256': '259e621a34e9effe09792e0cd20a1d54966e693440c7f000647e003765178c45',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, timm, lightning, tifffile
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'timm': timm.__version__, 'lightning': lightning.__version__, 'tifffile': tifffile.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/prithvi_burnscar_segmentation_pipeline/` @ `uncommitted`)

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/prithvi_burnscar_segmentation_pipeline/pipeline.py`

In [ ]:
"""Prithvi-EO-2.0-300M BurnScars (`ibm-nasa-geospatial/Prithvi-EO-2.0-300M-BurnScars`) DIMER pipeline: verified
snapshot, one-time conversion of the pickled Lightning checkpoint into safetensors, burn-scar segmentation of six-band
HLS chips, held-out evaluation against a not-burned baseline, and bounded fine-tuning of the decoder to a user's
labelled chips with a portable adapter.

Prithvi-EO-2.0 (Szwarcman et al., 2024) is a ViT-L masked-autoencoder foundation model for Harmonized Landsat
Sentinel-2 imagery. The checkpoint packaged here is the upstream authors' fine-tune for burn-scar mapping: the
300 M-parameter encoder (the variant without temporal/location embeddings), a `LearnedInterpolateToPyramidal` neck
over four encoder depths, a U-Net decoder (512 / 256 / 128 / 64 channels) and a two-class head, trained on the
804 512 × 512 HLS scenes of the HLS Burn Scars dataset (six bands: blue, green, red, narrow NIR, SWIR 1, SWIR 2)
with TerraTorch on the authors' non-overlapping splits.

The upstream asset is a PyTorch Lightning checkpoint — a torch zip archive whose pickle references only
`collections.OrderedDict`, `torch._utils._rebuild_tensor_v2` and two storage classes (verified statically by
`audit_pickle`). Under the fleet asset specification (§11) that is executable serialization, so this package
converts it once — `torch.load(weights_only=True)`, the `state_dict` entry, the `model.` prefix stripped — into
safetensors with a pinned digest, and serves only the converted file. The architecture is rebuilt from the
`terratorch` package on PyPI with `backbone_pretrained=False` and loaded strictly; nothing is fetched from the Hub
at load time except the manifest-listed files.

Everything model-related is imported lazily so that snapshot verification, the pickle audit and input validation
run (and can refuse) before `torch` or `terratorch` are imported (fleet RTM-001). `numpy` and `tifffile` are used
for chips and are imported freely.
"""

from __future__ import annotations

import hashlib
import io
import json
import math
import pickletools
import time
import warnings
import zipfile
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

MODEL_ID = "ibm-nasa-geospatial/Prithvi-EO-2.0-300M-BurnScars"
MODEL_REVISION = "a3f2c410e45b8ac7417976614528a872f024d831"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "prithvi-eo-2.0-300m-burnscars"
ARTIFACT_FORMAT = "org.valcorza.prithvi-burnscar-segmentation.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Immutable upstream source asset (a Lightning checkpoint, i.e. a pickle; see docs/WEIGHTS.md).
SOURCE_CKPT_NAME = "Prithvi_EO_V2_300M_BurnScars.pt"
SOURCE_CKPT_BYTES = 1_297_798_380
SOURCE_CKPT_SHA256 = "0c5f9334be9a75c9006387ab8f3dc05a55ea7fb5ef7956717316be57c62954d3"
# Code-free serving file produced deterministically by `convert_model` (asset spec §11.2).
CONVERTED_WEIGHTS_NAME = "prithvi-eo-2.0-300m-burnscars.safetensors"
CONVERTED_SHA256 = "4209c5a013f90dfe372abb4ade3865e7a283d655ef17b1c7de95d36ecc033c3f"
CONVERTED_BYTES = 1_297_682_024
# Static-audit digest of the source pickle (sorted global names), see `audit_pickle`.
PICKLE_AUDIT_SHA256 = "5b9f0ba08490293d6c17b9cef219991e1a6edda31609429679f8dca1af5a7b10"
CKPT_ALLOWED_GLOBALS = frozenset(
    {"collections.OrderedDict", "torch._utils._rebuild_tensor_v2", "torch.FloatStorage", "torch.LongStorage"}
)
STATE_DICT_PREFIX = "model."

# Architecture (burn_scars_config.yaml of the pinned snapshot) and data-contract facts.
BACKBONE = "prithvi_eo_v2_300"
DECODER = "UNetDecoder"
DECODER_ARGS: dict[str, Any] = {"decoder_channels": [512, 256, 128, 64]}
NECKS: tuple[dict[str, Any], ...] = (
    {"name": "SelectIndices", "indices": [5, 11, 17, 23]},
    {"name": "ReshapeTokensToImage"},
    {"name": "LearnedInterpolateToPyramidal"},
)
HEAD_ARGS: dict[str, Any] = {}
PARAMETER_COUNT = 324_204_674  # nn.Parameters; the state dict also carries 206,601 buffer elements
STATE_NUMEL = 324_411_275
STATE_TENSORS = 355
ENCODER_TENSORS = 294
NUM_CLASSES = 2
CLASS_NAMES: tuple[str, ...] = ("not burned", "burn scar")
IGNORE_INDEX = -1
BANDS: tuple[str, ...] = ("BLUE", "GREEN", "RED", "NIR_NARROW", "SWIR_1", "SWIR_2")
# A 13-band Sentinel-2 L1C stack, if given, is reduced to the six HLS-equivalent bands (B2, B3, B4, B8A, B11, B12).
S2_L1C_BAND_INDICES: tuple[int, ...] = (1, 2, 3, 8, 11, 12)
# The band statistics of the pinned burn_scars_config.yaml (reflectance scale).
MEANS: tuple[float, ...] = (
    0.033349706741586264,
    0.05701185520536176,
    0.05889748132001316,
    0.2323245113436119,
    0.1972854853760658,
    0.11944914225186566,
)
STDS: tuple[float, ...] = (
    0.02269135568823774,
    0.026807560223070237,
    0.04004109844362779,
    0.07791732423672691,
    0.08708738838140137,
    0.07241979477437814,
)
CONSTANT_SCALE = 1e-4  # applied only when a chip arrives as reflectance × 10 000; HLS scenes are already [0, 1]
NO_DATA_VALUES: tuple[float, ...] = (0.0, -9999.0)  # replaced by 0 before normalisation, as upstream
IMAGE_SIZE = 512
MIN_RECORDS = 4
MAX_RECORDS = 2_000
ADAPTATION_MODES = ("decoder", "decoder+last_block")  # the only scopes an adapter may declare
LAST_BLOCK_PREFIX = "encoder.blocks.23."
TRAINABLE_PREFIXES: dict[str, tuple[str, ...]] = {
    "decoder": ("neck.", "decoder.", "head."),
    "decoder+last_block": ("neck.", "decoder.", "head.", LAST_BLOCK_PREFIX),
}


# --------------------------------------------------------------------------------------------------
# manifest, staging, static pickle audit and conversion
# --------------------------------------------------------------------------------------------------


def _sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _verify_manifest(root: Path, model_id: str, revision: str) -> dict[str, Any]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"no snapshot manifest at {manifest_path}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("modelId") != model_id:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {model_id!r}")
    if manifest.get("revision") != revision:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {revision!r}")
    listed = {entry["path"] for entry in manifest["files"]}
    if SOURCE_CKPT_NAME not in listed:
        raise ValueError(f"manifest does not list {SOURCE_CKPT_NAME}; refusing to proceed")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256_file(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
        if entry["path"] == SOURCE_CKPT_NAME and (size, digest) != (SOURCE_CKPT_BYTES, SOURCE_CKPT_SHA256):
            raise ValueError(f"{entry['path']}: manifest digest disagrees with the package constant")
    return manifest


def verify_converted(path: str | Path | None = None) -> dict[str, Any]:
    """Check the converted serving file (safetensors) against the pinned digest."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    file_path = root / CONVERTED_WEIGHTS_NAME
    if not file_path.is_file():
        raise FileNotFoundError(f"converted file missing: {file_path}")
    size = file_path.stat().st_size
    if size != CONVERTED_BYTES:
        raise ValueError(f"{CONVERTED_WEIGHTS_NAME}: size {size} != pinned {CONVERTED_BYTES}")
    digest = _sha256_file(file_path)
    if digest != CONVERTED_SHA256:
        raise ValueError(f"{CONVERTED_WEIGHTS_NAME}: sha256 {digest} != pinned {CONVERTED_SHA256}")
    return {"files": [{"path": CONVERTED_WEIGHTS_NAME, "bytes": size, "sha256": digest}]}


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check the snapshot against its DIMER manifest (size + SHA-256 of every listed Hub file) and, when the
    converted serving file is present, that against the pinned digest."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _verify_manifest(root, MODEL_ID, MODEL_REVISION)
    converted = (root / CONVERTED_WEIGHTS_NAME).is_file()
    if converted:
        verify_converted(root)
    return {**manifest, "converted": converted}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at the pinned revision straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest entries that are absent locally (a fresh clone commits the manifest and git-ignores the
    1.28 GB checkpoint and the safetensors it converts to)."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def _pickle_globals(data: bytes) -> dict[str, int]:
    """Every global a pickle stream would import, collected with `pickletools.genops` (no execution)."""
    found: dict[str, int] = {}
    stack: list[Any] = []
    for op, arg, _pos in pickletools.genops(io.BytesIO(data)):
        if op.name == "GLOBAL":  # pickletools renders the (module, name) pair space-separated
            key = arg.replace("\n", " ").replace(" ", ".", 1)
            found[key] = found.get(key, 0) + 1
        elif op.name == "STACK_GLOBAL":
            key = f"{stack[-2]}.{stack[-1]}"
            found[key] = found.get(key, 0) + 1
        if op.name in ("SHORT_BINUNICODE", "BINUNICODE", "UNICODE", "SHORT_BINSTRING", "BINSTRING"):
            stack.append(arg)
        elif op.name in ("MEMOIZE", "BINPUT", "LONG_BINPUT", "PUT"):
            pass
        else:
            stack.append(None)
    return found


def audit_pickle(path: str | Path, *, allowed: frozenset[str] = CKPT_ALLOWED_GLOBALS) -> dict[str, Any]:
    """Statically list the globals a pickle (plain, or inside a torch zip archive) would import and refuse any
    outside `allowed`. Executes nothing. Returns the sorted globals and their digest."""
    file_path = Path(path)
    if not file_path.is_file():
        raise FileNotFoundError(f"file not found: {file_path}")
    data = file_path.read_bytes()
    found: dict[str, int] = {}
    nested = 0
    if data[:4] == b"PK\x03\x04":
        archive = zipfile.ZipFile(io.BytesIO(data))
        for name in archive.namelist():
            if name.endswith(".pkl"):
                nested += 1
                for key, count in _pickle_globals(archive.read(name)).items():
                    found[key] = found.get(key, 0) + count
    else:
        found = _pickle_globals(data)
    violations = sorted(name for name in found if name not in allowed)
    summary = {
        "file": file_path.name,
        "torch_archive": data[:4] == b"PK\x03\x04",
        "pickles": nested if nested else 1,
        "globals": sorted(found),
        "violations": violations,
        "audit_sha256": hashlib.sha256("\n".join(sorted(found)).encode("utf-8")).hexdigest(),
    }
    if violations:
        raise ValueError(f"{file_path.name}: pickle audit failed, globals outside the allow-list: {violations}")
    return summary


def _check_pinned_source(root: Path) -> dict[str, Any]:
    source = root / SOURCE_CKPT_NAME
    if not source.is_file():
        raise FileNotFoundError(f"source file not found: {source}")
    size = source.stat().st_size
    if size != SOURCE_CKPT_BYTES:
        raise ValueError(f"{SOURCE_CKPT_NAME}: size {size} != pinned {SOURCE_CKPT_BYTES}")
    digest = _sha256_file(source)
    if digest != SOURCE_CKPT_SHA256:
        raise ValueError(f"{SOURCE_CKPT_NAME}: sha256 {digest} != pinned {SOURCE_CKPT_SHA256}")
    audit = audit_pickle(source)
    if audit["audit_sha256"] != PICKLE_AUDIT_SHA256:
        raise ValueError(f"{SOURCE_CKPT_NAME}: pickle audit digest {audit['audit_sha256']} != pinned {PICKLE_AUDIT_SHA256}")
    return {"path": SOURCE_CKPT_NAME, "bytes": size, "sha256": digest, "audit": audit}


def build_model() -> Any:
    """Instantiate the fine-tuned architecture from the installed `terratorch` package (no pretrained download)."""
    from terratorch.models import EncoderDecoderFactory

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return EncoderDecoderFactory().build_model(
            task="segmentation",
            backbone=BACKBONE,
            backbone_pretrained=False,
            backbone_bands=list(BANDS),
            decoder=DECODER,
            num_classes=NUM_CLASSES,
            rescale=True,
            necks=[dict(n) for n in NECKS],
            **DECODER_ARGS,
            **HEAD_ARGS,
        )


def convert_model(path: str | Path | None = None) -> dict[str, Any]:
    """Convert the pinned Lightning checkpoint into safetensors, deterministically, after size, digest and
    static-audit checks: torch's weights-only unpickler, the `state_dict` entry with the `model.` prefix
    stripped, a strict load into the rebuilt architecture, and the model's own state dict saved."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    source = _check_pinned_source(root)
    import torch
    from safetensors.torch import save_file

    started = time.perf_counter()
    payload = torch.load(root / SOURCE_CKPT_NAME, map_location="cpu", weights_only=True)
    if not isinstance(payload, dict) or "state_dict" not in payload:
        raise ValueError(f"{SOURCE_CKPT_NAME} did not unpickle to a Lightning checkpoint with a state_dict")
    state = payload["state_dict"]
    if not isinstance(state, dict) or any(not isinstance(v, torch.Tensor) for v in state.values()):
        raise ValueError(f"{SOURCE_CKPT_NAME}: state_dict is not a dict of tensors")
    stripped = {}
    for key, value in state.items():
        if not key.startswith(STATE_DICT_PREFIX):
            raise ValueError(f"{SOURCE_CKPT_NAME}: unexpected state-dict key {key!r} outside {STATE_DICT_PREFIX!r}")
        stripped[key[len(STATE_DICT_PREFIX) :]] = value
    model = build_model()
    model.load_state_dict(stripped, strict=True)
    canonical = {k: v.contiguous() for k, v in model.state_dict().items()}
    n_params = sum(v.numel() for v in canonical.values())
    if len(canonical) != STATE_TENSORS or n_params != STATE_NUMEL:
        raise ValueError(
            f"converted state dict has {len(canonical)} tensors / {n_params} elements; expected {STATE_TENSORS} / {STATE_NUMEL}"
        )
    save_file(canonical, str(root / CONVERTED_WEIGHTS_NAME), metadata={"format": "pt"})
    report = verify_converted(root)
    return {
        "source": {k: v for k, v in source.items() if k != "audit"},
        "audit": source["audit"],
        "checkpoint": {
            "epoch": payload.get("epoch"),
            "global_step": payload.get("global_step"),
            "lightning_version": payload.get("pytorch-lightning_version"),
        },
        "converted": report["files"],
        "seconds": round(time.perf_counter() - started, 2),
    }


# --------------------------------------------------------------------------------------------------
# chips, labels and validation (no model import)
# --------------------------------------------------------------------------------------------------

INPUT_SCHEMA: dict[str, Any] = {
    "record": (
        "{id, image, label?}: image = (6, 512, 512) float32 reflectance chip (or a GeoTIFF path); "
        "label = (512, 512) int mask with 0/1/-1 (or a GeoTIFF path), optional"
    ),
    "bands": list(BANDS),
    "image_size": IMAGE_SIZE,
    "value_units": (
        "surface reflectance in [0, 1] (the HLS Burn Scars encoding, float32) or reflectance × 10 000; "
        "values above 1 are scaled by 1e-4"
    ),
    "no_data": list(NO_DATA_VALUES),
    "classes": {str(i): name for i, name in enumerate(CLASS_NAMES)},
    "ignore_index": IGNORE_INDEX,
    "records": [MIN_RECORDS, MAX_RECORDS],
    "validation": (
        "record shape, band count, chip size, finiteness, value range and label values only. Nothing checks that "
        "the bands are the six HLS bands in the right order, that the reflectance is atmospherically corrected, "
        "or that the label was drawn for this chip -- any six-band 512 × 512 array is segmented without complaint"
    ),
}


def _read_tiff(path: Path) -> Any:
    """Read a GeoTIFF's pixel array with tifffile as (bands, H, W) or (H, W); no georeferencing is used."""
    import numpy as np
    import tifffile

    with tifffile.TiffFile(path) as tf:
        array = tf.asarray()
        planar = tf.pages[0].planarconfig
    if array.ndim == 3 and planar is not None and int(planar) == 1 and array.shape[-1] <= 16:
        array = np.moveaxis(array, -1, 0)  # pixel-interleaved -> band-sequential
    return np.asarray(array)


def read_chip(path: str | Path, *, band_indices: Sequence[int] | None = None) -> Any:
    """Load a chip from a GeoTIFF as float32 (6, H, W); `band_indices` selects the six HLS bands from a wider
    stack (e.g. S2_L1C_BAND_INDICES for a 13-band Sentinel-2 L1C file)."""
    import numpy as np

    array = _read_tiff(Path(path))
    if array.ndim != 3:
        raise ValueError(f"{Path(path).name}: expected a multi-band raster, got shape {array.shape}")
    if band_indices is not None:
        array = array[list(band_indices)]
    elif array.shape[0] != len(BANDS) and array.shape[0] == 13:
        array = array[list(S2_L1C_BAND_INDICES)]
    return np.ascontiguousarray(array.astype(np.float32))


def read_mask(path: str | Path) -> Any:
    """Load a label raster from a GeoTIFF as int64 (H, W)."""
    import numpy as np

    array = _read_tiff(Path(path))
    if array.ndim == 3:
        if array.shape[0] != 1:
            raise ValueError(f"{Path(path).name}: a label raster must have one band, got shape {array.shape}")
        array = array[0]
    return np.ascontiguousarray(array.astype(np.int64))


def _check_record(record: Any, index: int) -> dict[str, Any]:
    import numpy as np

    label_name = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(f"{label_name} must be a mapping with id/image[/label]")
    for key in ("id", "image"):
        if key not in record:
            raise ValueError(f"{label_name} is missing {key!r}")
    rid, image = record["id"], record["image"]
    if not isinstance(rid, str) or not rid or len(rid) > 128:
        raise ValueError(f"{label_name}: id must be a non-empty string of at most 128 characters")
    if isinstance(image, str | Path):
        if not Path(image).is_file():
            raise ValueError(f"{label_name}: image file not found: {image}")
        image = read_chip(image)
    try:
        array = np.asarray(image, dtype=np.float32)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{label_name}: image must be a numeric array") from exc
    if array.shape != (len(BANDS), IMAGE_SIZE, IMAGE_SIZE):
        raise ValueError(f"{label_name}: image must have shape {(len(BANDS), IMAGE_SIZE, IMAGE_SIZE)}, got {array.shape}")
    if not np.all(np.isfinite(array)):
        raise ValueError(f"{label_name}: image contains non-finite values")
    for value in NO_DATA_VALUES:
        array = np.where(array == value, 0.0, array)
    if float(array.max()) > 1.0:
        array = array * CONSTANT_SCALE
    if float(array.min()) < -0.5 or float(array.max()) > 2.0:
        span = (float(array.min()), float(array.max()))
        raise ValueError(f"{label_name}: reflectance outside the plausible range after scaling: {span}")
    item: dict[str, Any] = {"id": rid, "image": np.ascontiguousarray(array.astype(np.float32))}
    label = record.get("label")
    if label is not None:
        if isinstance(label, str | Path):
            if not Path(label).is_file():
                raise ValueError(f"{label_name}: label file not found: {label}")
            label = read_mask(label)
        try:
            mask = np.asarray(label)
        except (TypeError, ValueError) as exc:
            raise ValueError(f"{label_name}: label must be an integer array") from exc
        if mask.shape != (IMAGE_SIZE, IMAGE_SIZE):
            raise ValueError(f"{label_name}: label must have shape {(IMAGE_SIZE, IMAGE_SIZE)}, got {mask.shape}")
        if not np.issubdtype(mask.dtype, np.integer) and not np.all(mask == np.round(mask)):
            raise ValueError(f"{label_name}: label values must be integers")
        allowed = set(range(NUM_CLASSES)) | {IGNORE_INDEX}
        found = set(np.unique(mask).astype(int).tolist())
        if not found <= allowed:
            raise ValueError(f"{label_name}: label values {sorted(found - allowed)} outside {sorted(allowed)}")
        item["label"] = np.ascontiguousarray(mask.astype(np.int64))
    for key in ("split", "region", "source", "source_id"):
        if key in record:
            item[key] = record[key]
    return item


def check_record(record: Mapping[str, Any]) -> dict[str, Any]:
    """Validate one record and return its normalised copy (float32 reflectance, no-data replaced, int64 label)."""
    return _check_record(record, 0)


def chip_digest(record: Mapping[str, Any]) -> str:
    checked = _check_record(record, 0)
    digest = hashlib.sha256(checked["image"].tobytes())
    if "label" in checked:
        digest.update(checked["label"].tobytes())
    return digest.hexdigest()


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    payload = [[r["id"], chip_digest(r)] for r in records]
    return hashlib.sha256(json.dumps(payload, separators=(",", ":")).encode("utf-8")).hexdigest()


def validate_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    min_records: int = MIN_RECORDS,
    max_records: int = MAX_RECORDS,
    require_labels: bool = True,
) -> dict[str, Any]:
    """Structural validation of a chip dataset; raises ValueError before any model import."""
    import numpy as np

    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, str | bytes):
        raise ValueError("records must be a list of {id, image, label} mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    checked = []
    ids: set[str] = set()
    class_pixels = np.zeros(NUM_CLASSES, dtype=np.int64)
    ignored = 0
    for index, record in enumerate(records):
        item = _check_record(record, index)
        if item["id"] in ids:
            raise ValueError(f"duplicate id {item['id']!r}")
        ids.add(item["id"])
        if require_labels and "label" not in item:
            raise ValueError(f"records[{index}] has no label; every record of a labelled dataset needs one")
        if "label" in item:
            for c in range(NUM_CLASSES):
                class_pixels[c] += int((item["label"] == c).sum())
            ignored += int((item["label"] == IGNORE_INDEX).sum())
        checked.append(item)
    labelled = sum("label" in r for r in checked)
    if require_labels and labelled and class_pixels[1] == 0:
        raise ValueError(f"no pixel of class 1 ({CLASS_NAMES[1]}) in the dataset; nothing to learn or evaluate")
    total = int(class_pixels.sum())
    return {
        "records": checked,
        "n_records": len(checked),
        "n_labelled": labelled,
        "image_size": IMAGE_SIZE,
        "bands": list(BANDS),
        "class_pixel_fraction": {
            CLASS_NAMES[c]: round(float(class_pixels[c]) / total, 4) if total else None for c in range(NUM_CLASSES)
        },
        "ignored_pixels": ignored,
        "reflectance_range": [
            round(float(min(r["image"].min() for r in checked)), 4),
            round(float(max(r["image"].max() for r in checked)), 4),
        ],
        "digest": dataset_digest(checked),
        "model_id": MODEL_ID,
    }


def validate_inputs(record: Mapping[str, Any]) -> dict[str, Any]:
    """Validate one record; returns its id, shape, reflectance range and label class fractions."""
    item = _check_record(record, 0)
    report = {
        "id": item["id"],
        "shape": tuple(item["image"].shape),
        "reflectance_range": [round(float(item["image"].min()), 4), round(float(item["image"].max()), 4)],
        "has_label": "label" in item,
    }
    if "label" in item:
        label = item["label"]
        valid = int((label != IGNORE_INDEX).sum())
        report["label_fraction"] = {
            CLASS_NAMES[c]: round(float((label == c).sum()) / max(valid, 1), 4) for c in range(NUM_CLASSES)
        }
        report["ignored_pixels"] = int((label == IGNORE_INDEX).sum())
    return report


# --------------------------------------------------------------------------------------------------
# pipeline
# --------------------------------------------------------------------------------------------------


def _normalise(images: Any) -> Any:
    """(B, 6, H, W) reflectance -> standardised with the upstream datamodule statistics."""
    import numpy as np

    mean = np.asarray(MEANS, dtype=np.float32)[None, :, None, None]
    std = np.asarray(STDS, dtype=np.float32)[None, :, None, None]
    return (images - mean) / std


@dataclass
class PrithviBurnScarPipeline:
    """Burn-scar segmentation and bounded decoder fine-tuning on top of the verified Prithvi burn-scar model."""

    model: Any
    device: str
    weights_dir: Path
    source: str
    adapter: dict[str, Any] | None = None

    @classmethod
    def from_pretrained(
        cls,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
        require_source: bool = True,
        report: Callable[[dict[str, Any]], None] | None = None,
    ) -> PrithviBurnScarPipeline:
        """Verify, convert if needed, rebuild from the installed package and strictly load. With
        `require_source=False` the checkpoint may be absent (the DIMER-hosted case) as long as the converted file
        verifies. `report` receives the audit and conversion records when a conversion happens."""
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if require_source:
            stage_missing_files(root, allow_download=allow_download)
            snapshot = verify_snapshot(root)
            if not snapshot["converted"]:
                conversion = convert_model(root)
                if report is not None:
                    report({"conversion": conversion})
                snapshot = verify_snapshot(root)
            elif report is not None:
                report({"conversion": "converted file already present and digest-verified"})
            source = "converted from the manifest-verified source checkpoint"
        else:
            verify_converted(root)
            source = "converted file, pinned digest (source checkpoint not required)"
        import torch
        from safetensors.torch import load_file

        chosen = device or ("cuda" if torch.cuda.is_available() else "cpu")
        if chosen.startswith("cuda") and not torch.cuda.is_available():
            raise ValueError("device='cuda' requested but CUDA is not available")
        model = build_model()
        state = load_file(str(root / CONVERTED_WEIGHTS_NAME))
        model.load_state_dict(state, strict=True)
        n_params = sum(p.numel() for p in model.parameters())
        if n_params != PARAMETER_COUNT:
            raise ValueError(f"rebuilt model has {n_params} parameters, expected {PARAMETER_COUNT}")
        model.to(torch.device(chosen)).eval()
        for param in model.parameters():
            param.requires_grad_(False)
        return cls(model=model, device=chosen, weights_dir=root, source=source)

    # ---- forward ---------------------------------------------------------------------------------------

    def _logits(self, images: Any, *, grad: bool = False) -> Any:
        """(B, 6, H, W) float32 reflectance -> (B, 2, H, W) float32 logits at input resolution."""
        import numpy as np
        import torch

        batch = torch.from_numpy(_normalise(np.asarray(images, dtype=np.float32))).to(self.device)
        use_amp = self.device.startswith("cuda")
        context = torch.enable_grad() if grad else torch.inference_mode()
        with context, torch.autocast(device_type=self.device.split(":")[0], dtype=torch.float16, enabled=use_amp):
            out = self.model(batch)
        logits = out.output if hasattr(out, "output") else out
        if tuple(logits.shape[-2:]) != tuple(batch.shape[-2:]):
            logits = torch.nn.functional.interpolate(logits.float(), size=batch.shape[-2:], mode="bilinear", align_corners=False)
        return logits.float()

    # ---- inference -------------------------------------------------------------------------------------

    def predict(self, records: Sequence[Mapping[str, Any]], *, batch_size: int = 4) -> dict[str, Any]:
        """Segment chips: per record the argmax mask (H, W) uint8, the softmax scores (2, H, W) float32 and the
        fraction of pixels in each class. Softmax scores are the model's own outputs, not calibrated probabilities."""
        import numpy as np
        import torch

        checked = validate_dataset(records, min_records=1, require_labels=False)["records"]
        if not isinstance(batch_size, int) or not 1 <= batch_size <= 32:
            raise ValueError("batch_size must be an int in 1..32")
        started = time.perf_counter()
        predictions = []
        for start in range(0, len(checked), batch_size):
            batch = checked[start : start + batch_size]
            logits = self._logits(np.stack([r["image"] for r in batch]))
            scores = torch.softmax(logits, dim=1).cpu().numpy()
            masks = scores.argmax(axis=1).astype(np.uint8)
            for record, score, mask in zip(batch, scores, masks, strict=True):
                predictions.append(
                    {
                        "id": record["id"],
                        "mask": mask,
                        "scores": score.astype(np.float32),
                        "class_fraction": {CLASS_NAMES[c]: round(float((mask == c).mean()), 4) for c in range(NUM_CLASSES)},
                    }
                )
        return {
            "model": {"id": MODEL_ID, "revision": MODEL_REVISION, "key": MODEL_KEY, "adapted": self.adapter is not None},
            "classes": list(CLASS_NAMES),
            "decision_rule": "argmax over the two class scores (no threshold)",
            "predictions": predictions,
            "seconds": round(time.perf_counter() - started, 3),
        }

    def evaluate(self, records: Sequence[Mapping[str, Any]], *, batch_size: int = 4) -> dict[str, Any]:
        """Pixel-level metrics on labelled chips (ignore index excluded): per-class IoU, mean IoU, accuracy and the
        class-1 F1, precision and recall, with the all-not-burned baseline scored on the same pixels."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import majority_baseline, segmentation_metrics` removed — names are kernel globals defined by the carried modules

        checked = validate_dataset(records, min_records=1)["records"]
        started = time.perf_counter()
        result = self.predict(checked, batch_size=batch_size)
        masks = [p["mask"] for p in result["predictions"]]
        labels = [r["label"] for r in checked]
        metrics = segmentation_metrics(masks, labels)
        return {
            "n_records": len(checked),
            "metric": "pixel IoU / F1 over the labelled pixels of the held-out chips (ignore index excluded)",
            "model": metrics,
            "baseline_not_burned": majority_baseline(labels),
            "adapted": self.adapter is not None,
            "seconds": round(time.perf_counter() - started, 3),
        }

    # ---- adaptation ------------------------------------------------------------------------------------

    def _trainable(self, mode: str) -> list[str]:
        if mode not in ADAPTATION_MODES:
            raise ValueError(f"trainable must be one of {ADAPTATION_MODES}")
        prefixes = TRAINABLE_PREFIXES[mode]
        return sorted(name for name, _param in self.model.named_parameters() if name.startswith(prefixes))

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None = None,
        *,
        epochs: int = 4,
        lr: float = 1e-5,
        batch_size: int = 2,
        trainable: str = "decoder",
        seed: int = 0,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded fine-tuning of the neck, decoder and head (`trainable="decoder"`; `"decoder+last_block"` also
        unfreezes the last encoder block) on labelled chips: cross-entropy over the labelled pixels (ignore index
        excluded), AdamW at a fixed learning rate, seeded horizontal/vertical flips, float16 autocast with loss
        scaling on CUDA. Epoch 0 records the frozen model; the epoch with the lowest validation loss is kept."""
        if not isinstance(epochs, int) or not 1 <= epochs <= 50:
            raise ValueError("epochs must be an int in 1..50")
        if not (0.0 < lr <= 1e-2):
            raise ValueError("lr must be in (0, 1e-2]")
        if not isinstance(batch_size, int) or not 1 <= batch_size <= 16:
            raise ValueError("batch_size must be an int in 1..16")
        names = self._trainable(trainable)
        train_checked = validate_dataset(train)["records"]
        val_checked = validate_dataset(val, min_records=1)["records"] if val is not None else None
        import numpy as np
        import torch

        torch.manual_seed(seed)
        started = time.perf_counter()
        model = self.model
        name_set = set(names)
        for name, param in model.named_parameters():
            param.requires_grad_(name in name_set)
        params = [p for n, p in model.named_parameters() if n in name_set]
        n_trainable = sum(p.numel() for p in params)
        optimiser = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
        use_amp = self.device.startswith("cuda")
        scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
        rng = np.random.default_rng(seed)

        def val_loss() -> float | None:
            if val_checked is None:
                return None
            model.eval()
            losses = []
            for start in range(0, len(val_checked), batch_size):
                batch = val_checked[start : start + batch_size]
                logits = self._logits(np.stack([r["image"] for r in batch]))
                target = torch.from_numpy(np.stack([r["label"] for r in batch])).to(self.device)
                losses.append(float(torch.nn.functional.cross_entropy(logits, target, ignore_index=IGNORE_INDEX)))
            return sum(losses) / len(losses)

        initial_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in name_set}
        try:
            history: list[dict[str, Any]] = []
            entry: dict[str, Any] = {"epoch": 0, "train_loss": None, "val_loss": val_loss(), "note": "frozen model"}
            if val_checked is not None:
                entry["val"] = self.evaluate(val_checked, batch_size=batch_size)["model"]
            history.append(entry)
            best_val = entry["val_loss"] if entry["val_loss"] is not None else math.inf
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in name_set}
            best_epoch = 0
            if progress:
                progress(entry)
            n_steps = 0
            for epoch in range(1, epochs + 1):
                model.train()
                for module in model.modules():  # BatchNorm statistics stay frozen: tiny batches would corrupt them
                    if isinstance(module, torch.nn.modules.batchnorm._BatchNorm):
                        module.eval()
                order = rng.permutation(len(train_checked)).tolist()
                losses = []
                for start in range(0, len(order), batch_size):
                    batch = [train_checked[i] for i in order[start : start + batch_size]]
                    images = np.stack([r["image"] for r in batch])
                    labels = np.stack([r["label"] for r in batch])
                    if rng.random() < 0.5:
                        images, labels = images[..., ::-1], labels[..., ::-1]
                    if rng.random() < 0.5:
                        images, labels = images[..., ::-1, :], labels[..., ::-1, :]
                    logits = self._logits(np.ascontiguousarray(images), grad=True)
                    target = torch.from_numpy(np.ascontiguousarray(labels)).to(self.device)
                    loss = torch.nn.functional.cross_entropy(logits, target, ignore_index=IGNORE_INDEX)
                    optimiser.zero_grad(set_to_none=True)
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimiser)
                    torch.nn.utils.clip_grad_norm_(params, 1.0)
                    scaler.step(optimiser)
                    scaler.update()
                    losses.append(float(loss.detach()))
                    n_steps += 1
                model.eval()
                entry = {"epoch": epoch, "train_loss": sum(losses) / len(losses), "val_loss": val_loss()}
                if val_checked is not None:
                    entry["val"] = self.evaluate(val_checked, batch_size=batch_size)["model"]
                history.append(entry)
                if progress:
                    progress(entry)
                if entry["val_loss"] is None or entry["val_loss"] < best_val:
                    best_val = entry["val_loss"] if entry["val_loss"] is not None else best_val
                    best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in name_set}
                    best_epoch = epoch
        except BaseException:
            # Transactional: a failure in training, validation or the progress callback leaves the model as it
            # was before adapt() (trained tensors restored), frozen, with no adapter attached.
            restore = dict(model.state_dict())
            restore.update(initial_state)
            model.load_state_dict(restore, strict=True)
            model.eval()
            for param in model.parameters():
                param.requires_grad_(False)
            self.adapter = None
            raise
        merged = dict(model.state_dict())
        merged.update(best_state)
        model.load_state_dict(merged, strict=True)
        model.eval()
        for param in model.parameters():
            param.requires_grad_(False)
        self.adapter = {
            "trainable": trainable,
            "trainable_names": names,
            "n_trainable": n_trainable,
            "n_total": sum(p.numel() for p in model.parameters()),
            "epochs": epochs,
            "best_epoch": best_epoch,
            "lr": lr,
            "batch_size": batch_size,
            "augmentation": "seeded horizontal/vertical flips",
            "batchnorm": "running statistics frozen (eval mode) during adaptation",
            "precision": "float16 autocast + GradScaler" if use_amp else "float32",
            "n_train_records": len(train_checked),
            "n_steps": n_steps,
            "seed": seed,
            "history": history,
            "seconds": round(time.perf_counter() - started, 2),
        }
        return dict(self.adapter)

    # ---- artifacts -------------------------------------------------------------------------------------

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the adapted tensors as safetensors with a manifest."""
        if self.adapter is None:
            raise ValueError("nothing to save: call adapt() first")
        from safetensors.torch import save_file

        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = set(self.adapter["trainable_names"])
        tensors = {k: v.detach().cpu().contiguous() for k, v in self.model.state_dict().items() if k in names}
        weights_path = out / ARTIFACT_WEIGHTS_NAME
        save_file(tensors, str(weights_path), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {"id": MODEL_ID, "revision": MODEL_REVISION, "key": MODEL_KEY, "converted_sha256": CONVERTED_SHA256},
            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},
            "history": self.adapter["history"],
            "tensors": sorted(tensors),
            "files": [
                {"path": ARTIFACT_WEIGHTS_NAME, "bytes": weights_path.stat().st_size, "sha256": _sha256_file(weights_path)}
            ],
            "metadata": dict(metadata or {}),
        }
        (out / ARTIFACT_MANIFEST_NAME).write_text(json.dumps(manifest, indent=2), encoding="utf-8")
        return out

    @staticmethod
    def check_artifact_manifest(root: Path, manifest: Mapping[str, Any]) -> tuple[Path, str]:
        """Static checks on an adapter manifest, before any model or weights work: format and version, the pinned
        base and converted digest, exactly one weights entry named `adapter.safetensors` inside the artifact
        directory, and an adaptation mode that is one of the declared scopes. Returns the weights path and mode."""
        if manifest.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
        if manifest.get("format_version") != ARTIFACT_FORMAT_VERSION:
            raise ValueError(
                f"artifact format_version {manifest.get('format_version')!r} is not supported "
                f"(expected {ARTIFACT_FORMAT_VERSION!r})"
            )
        base = manifest.get("base_model", {})
        if (base.get("id"), base.get("revision")) != (MODEL_ID, MODEL_REVISION):
            raise ValueError("artifact was adapted from a different base model or revision")
        if base.get("converted_sha256") != CONVERTED_SHA256:
            raise ValueError("artifact records a different converted-base digest")
        files = manifest.get("files")
        if not isinstance(files, list) or len(files) != 1:
            raise ValueError("artifact manifest must list exactly one weights file")
        entry = files[0]
        if not isinstance(entry, Mapping) or entry.get("path") != ARTIFACT_WEIGHTS_NAME:
            raise ValueError(f"artifact weights file must be named {ARTIFACT_WEIGHTS_NAME!r}")
        weights_path = (root / entry["path"]).resolve()
        if weights_path.parent != root.resolve():
            raise ValueError("artifact weights file must sit inside the artifact directory")
        adapter = manifest.get("adapter")
        mode = adapter.get("trainable") if isinstance(adapter, Mapping) else None
        if mode not in ADAPTATION_MODES:
            raise ValueError(f"artifact adapter.trainable must be one of {ADAPTATION_MODES}")
        if not isinstance(manifest.get("tensors"), list):
            raise ValueError("artifact manifest must list its tensors")
        return weights_path, mode

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Verify an adapter's manifest, scope and digest, then overwrite exactly the tensors the scope allows."""
        root = Path(artifact_dir)
        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
        weights_path, mode = self.check_artifact_manifest(root, manifest)
        expected = self._trainable(mode)
        if sorted(manifest["tensors"]) != expected:
            raise ValueError(
                f"artifact tensor list does not match the {len(expected)} tensors that trainable={mode!r} may change"
            )
        entry = manifest["files"][0]
        if _sha256_file(weights_path) != entry["sha256"] or weights_path.stat().st_size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: digest or size mismatch; refusing to load")
        from safetensors.torch import load_file

        tensors = load_file(str(weights_path))
        if sorted(tensors) != expected:
            raise ValueError("artifact tensor names differ from the validated manifest")
        state = self.model.state_dict()
        for key, value in tensors.items():
            if tuple(value.shape) != tuple(state[key].shape):
                raise ValueError(f"artifact tensor {key} has shape {tuple(value.shape)}, model has {tuple(state[key].shape)}")
        merged = dict(state)
        merged.update({k: v.to(state[k].device, state[k].dtype) for k, v in tensors.items()})
        self.model.load_state_dict(merged, strict=True)
        self.model.eval()
        self.adapter = {**manifest["adapter"], "trainable_names": manifest["tensors"], "history": manifest.get("history", [])}
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
        require_source: bool = True,
    ) -> PrithviBurnScarPipeline:
        root = Path(artifact_dir)
        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
        cls.check_artifact_manifest(root, manifest)
        pipeline = cls.from_pretrained(
            device=device, weights_dir=weights_dir, allow_download=allow_download, require_source=require_source
        )
        pipeline.load_artifact(artifact_dir)
        return pipeline

**Module 2/3:** `src/prithvi_burnscar_segmentation_pipeline/metrics.py` (carried verbatim; see the note above)

In [ ]:
"""Pixel-level segmentation metrics from a confusion matrix over the labelled pixels (ignore index excluded):
per-class IoU, mean IoU, overall accuracy, and the positive class's precision, recall and F1 — plus the
"not burned" baseline that predicts class 0 everywhere, scored on exactly the same pixels.
"""

from __future__ import annotations

from collections.abc import Sequence
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import CLASS_NAMES, IGNORE_INDEX, NUM_CLASSES` removed — names are kernel globals defined by the carried modules


def confusion_matrix(predictions: Sequence[Any], labels: Sequence[Any]) -> Any:
    """(NUM_CLASSES, NUM_CLASSES) counts, rows = truth, columns = prediction; ignore-index pixels are skipped."""
    import numpy as np

    if len(predictions) != len(labels) or not labels:
        raise ValueError("predictions and labels must be non-empty sequences of equal length")
    matrix = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    for pred, label in zip(predictions, labels, strict=True):
        pred = np.asarray(pred).astype(np.int64)
        label = np.asarray(label).astype(np.int64)
        if pred.shape != label.shape:
            raise ValueError(f"prediction shape {pred.shape} != label shape {label.shape}")
        keep = label != IGNORE_INDEX
        if not np.all((pred[keep] >= 0) & (pred[keep] < NUM_CLASSES)) or not np.all(label[keep] < NUM_CLASSES):
            raise ValueError("class ids outside 0..NUM_CLASSES-1")
        matrix += np.bincount(label[keep] * NUM_CLASSES + pred[keep], minlength=NUM_CLASSES**2).reshape(NUM_CLASSES, NUM_CLASSES)
    return matrix


def metrics_from_confusion(matrix: Any) -> dict[str, Any]:
    import numpy as np

    matrix = np.asarray(matrix, dtype=np.float64)
    total = matrix.sum()
    if total == 0:
        raise ValueError("no labelled pixel to score")
    tp = np.diag(matrix)
    fp = matrix.sum(axis=0) - tp
    fn = matrix.sum(axis=1) - tp
    union = tp + fp + fn
    iou = np.where(union > 0, tp / np.maximum(union, 1), np.nan)
    present = matrix.sum(axis=1) > 0
    pos = NUM_CLASSES - 1
    precision = tp[pos] / (tp[pos] + fp[pos]) if tp[pos] + fp[pos] > 0 else 0.0
    recall = tp[pos] / (tp[pos] + fn[pos]) if tp[pos] + fn[pos] > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
    return {
        "iou": {CLASS_NAMES[c]: (round(float(iou[c]), 4) if not np.isnan(iou[c]) else None) for c in range(NUM_CLASSES)},
        "mean_iou": round(float(np.nanmean(iou[present])), 4),
        "accuracy": round(float(tp.sum() / total), 4),
        "precision": round(float(precision), 4),
        "recall": round(float(recall), 4),
        "f1": round(float(f1), 4),
        "positive_class": CLASS_NAMES[pos],
        "labelled_pixels": int(total),
        "positive_fraction": round(float(matrix[pos].sum() / total), 4),
        "confusion": matrix.astype(int).tolist(),
    }


def segmentation_metrics(predictions: Sequence[Any], labels: Sequence[Any]) -> dict[str, Any]:
    """Metrics of predicted masks against labels over all chips at once (pixel-pooled, not chip-averaged)."""
    return metrics_from_confusion(confusion_matrix(predictions, labels))


def majority_baseline(labels: Sequence[Any]) -> dict[str, Any]:
    """The all-class-0 prediction scored on the same pixels: the number any model must beat on the positive class."""
    import numpy as np

    predictions = [np.zeros_like(np.asarray(label), dtype=np.int64) for label in labels]
    report = segmentation_metrics(predictions, labels)
    report["note"] = f"predicts '{CLASS_NAMES[0]}' for every pixel"
    return report

**Module 3/3:** `src/prithvi_burnscar_segmentation_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Labelled-chip dataset contract for adapting the burn-scar model: the pinned HLS Burn Scars sample, role
assignment from the model repository's splits, BYOD loaders and sample export.

The default dataset is **real**: 44 labelled 512 × 512 HLS scenes of the HLS Burn Scars dataset (NASA IMPACT /
UAH, CC BY 4.0) — 24 from the training split, 8 from the validation split and 12 from the test split that the
upstream authors published beside the checkpoint (non-overlapping; the test split is what their reported IoU was
measured on), drawn with a fixed seed on 2026-09-19 from the scenes whose mask is at least 60 % valid and at least
3 % burn scar, so every chip can be scored. The dataset is distributed as one 2.6 GB gzipped tarball on the Hugging
Face Hub; the tarball is pinned by byte size and SHA-256, each pinned member is pinned again by size and SHA-256
and extracted **without** `extractall` into the cache, and everything else in the archive is left alone. The
repository redistributes none of the scenes.

A record is ``{id, image, label}``: a (6, 512, 512) reflectance array (or a GeoTIFF path) and a (512, 512) mask
with 0 = not burned, 1 = burn scar, -1 = no data (or a GeoTIFF path).
"""

from __future__ import annotations

import csv
import hashlib
import io
import json
import tarfile
import zipfile
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import (` removed — names are kernel globals defined by the carried modules

CORPUS_NAME = "HLS Burn Scars scenes (NASA IMPACT / UAH)"
CORPUS_RELEASE = (
    "Hugging Face dataset ibm-nasa-geospatial/hls_burn_scars, 44 scenes selected 2026-09-19 from the model repository's splits"
)
CORPUS_LICENSE = "CC BY 4.0 (NASA IMPACT / University of Alabama in Huntsville)"
DATASET_ID = "ibm-nasa-geospatial/hls_burn_scars"
DATASET_REVISION = "1864285e25010d346a842e4f068b1a1d4248ed6d"
CORPUS_BASE_URL = f"https://huggingface.co/datasets/{DATASET_ID}/resolve/{DATASET_REVISION}/"
TAR_NAME = "hls_burn_scars.tar.gz"
TAR_BYTES = 2_645_552_531
TAR_SHA256 = "4e6f99a75cb2c500547b20662a15cbd531dc421376f815e91846ea542798e8e6"
CORPUS_BYTES = 300_099_360  # the 88 pinned members, uncompressed
DEFAULT_CACHE_DIR = Path("weights") / "hls-burn-scars"
ROLES = ("train", "validation", "test")
# (scene key, role from the model repository's splits, image member, image bytes, image sha256, mask member,
#  mask bytes, mask sha256) — member paths are relative to the tarball root
SAMPLE_RECORDS: tuple[tuple[str, str, str, int, str, str, int, str], ...] = (
    (
        "T10SFE.2020267.v1",
        "train",
        "training/subsetted_512x512_HLS.S30.T10SFE.2020267.v1.4_merged.tif",
        6295168,
        "98e30b354bbceed46d3e6e1d2c3178c6fba27bf13f9a231ad12175c2374de16b",
        "training/subsetted_512x512_HLS.S30.T10SFE.2020267.v1.4.mask.tif",
        525272,
        "2b615ce4738332d4ccce0aa30a45254c828b28c2f68f2b73a395394a6d813eff",
    ),
    (
        "T10SGE.2019187.v1",
        "train",
        "training/subsetted_512x512_HLS.S30.T10SGE.2019187.v1.4_merged.tif",
        6295168,
        "d7fe88a2751e33808afc5fe07732e87c6028d09be3cceda5fe6eb17a4061feda",
        "training/subsetted_512x512_HLS.S30.T10SGE.2019187.v1.4.mask.tif",
        525272,
        "71596c6a730fe28e430ce457ee293039e7311ffa3f1db3d99438dcae754917cc",
    ),
    (
        "T10SGE.2020162.v1",
        "train",
        "training/subsetted_512x512_HLS.S30.T10SGE.2020162.v1.4_merged.tif",
        6295168,
        "a6fc23eeb0c070f223ad2eee8a218eab43c74f62d88316e3d0f3db48803c9f37",
        "training/subsetted_512x512_HLS.S30.T10SGE.2020162.v1.4.mask.tif",
        525272,
        "447d5416dc961f5d49a99a07ec44ca3f5149ea07f95604898d214a526a252994",
    ),
    (
        "T10SGG.2020247.v1",
        "train",
        "validation/subsetted_512x512_HLS.S30.T10SGG.2020247.v1.4_merged.tif",
        6295168,
        "51baf73bf405eefff177adc082fcfedbede30f50f92abdf8458880aa89aabeee",
        "validation/subsetted_512x512_HLS.S30.T10SGG.2020247.v1.4.mask.tif",
        525272,
        "2e661b9a46b70c275f0ae3e22594031c56c8c463b288e941159cf628872552c2",
    ),
    (
        "T10TFN.2018245.v1",
        "train",
        "training/subsetted_512x512_HLS.S30.T10TFN.2018245.v1.4_merged.tif",
        6295168,
        "5b8ca4c160402252d1c6f373ebee7ea9e197337bf6b768b20d6f10b266d64f49",
        "training/subsetted_512x512_HLS.S30.T10TFN.2018245.v1.4.mask.tif",
        525272,
        "668ce579477b7dc5c7dc537a4bd44ff6cc329d2f31b084f005f9cb7432efa1f2",
    ),
    (
        "T10TFQ.2019245.v1",
        "train",
        "training/subsetted_512x512_HLS.S30.T10TFQ.2019245.v1.4_merged.tif",
        6295168,
        "3b4f9f73d8b0b952bca710b96e4cac6d543baa95fe9dc7ee54c7479702e33704",
        "training/subsetted_512x512_HLS.S30.T10TFQ.2019245.v1.4.mask.tif",
        525272,
        "60c898f6f9ecc5710c934ebb5f576a96e1a181cee2fbd3c2d51e2c7a1e27afa5",
    ),
    (
        "T10TGS.2018190.v1",
        "train",
        "training/subsetted_512x512_HLS.S30.T10TGS.2018190.v1.4_merged.tif",
        6295168,
        "2ffd9a5805676b01463aa1df92c1f2e65f7b7c039c90e4a7500a38910ffc6e89",
        "training/subsetted_512x512_HLS.S30.T10TGS.2018190.v1.4.mask.tif",
        525272,
        "1e7480cdb18b3c8d51fd78dd440c434e72d96a713c19276f85632a20054e72f8",
    ),
    (
        "T10UGU.2018245.v1",
        "train",
        "training/subsetted_512x512_HLS.S30.T10UGU.2018245.v1.4_merged.tif",
        6295168,
        "fc44dbe9277a06793f8abac67a775d3bb9eb07d0bb4b31e3fb801196e8f62565",
        "training/subsetted_512x512_HLS.S30.T10UGU.2018245.v1.4.mask.tif",
        525272,
        "fd0d2bce787d50350f10e4fb194885d4f70f6c4397595b5db2a86f7bf53af2eb",
    ),
    (
        "T11SMT.2019294.v1",
        "train",
        "training/subsetted_512x512_HLS.S30.T11SMT.2019294.v1.4_merged.tif",
        6295168,
        "bf653eb96ee9bd8543ea06e5477be7fd2131d6403872bfcba78ea66423df6805",
        "training/subsetted_512x512_HLS.S30.T11SMT.2019294.v1.4.mask.tif",
        525272,
        "e479eb9e3d16dcd7ecfffeb3c017347222009cb74c95cb44e8785f0606363130",
    ),
    (
        "T11TPE.2019269.v1",
        "train",
        "training/subsetted_512x512_HLS.S30.T11TPE.2019269.v1.4_merged.tif",
        6295168,
        "3813f554b8b2d82bc7a26c3fd5e382cd904e4a72f8e7051cd2465bd04c43377d",
        "training/subsetted_512x512_HLS.S30.T11TPE.2019269.v1.4.mask.tif",
        525272,
        "a2876c506fe075263abce00ee409a12b3b888930e764f1147c32f66e85b476ba",
    ),
    (
        "T12STG.2018186.v1",
        "train",
        "validation/subsetted_512x512_HLS.S30.T12STG.2018186.v1.4_merged.tif",
        6295168,
        "6d9ff3c80912699273588a5c13468c7f9b977277d0da23485dd180b047c59ab8",
        "validation/subsetted_512x512_HLS.S30.T12STG.2018186.v1.4.mask.tif",
        525272,
        "799340ad87a0fcacddeedaeff2256ec898ac249dea5475e6a540e39f2907ee10",
    ),
    (
        "T12SYG.2018225.v1",
        "train",
        "training/subsetted_512x512_HLS.S30.T12SYG.2018225.v1.4_merged.tif",
        6295168,
        "7d81640e6ac0a6170a8d02d50cbdd2cc237fc9f29e7dbbd7d3294e4487bf7cb4",
        "training/subsetted_512x512_HLS.S30.T12SYG.2018225.v1.4.mask.tif",
        525272,
        "5efc42d679955ff8c665e95fcf15d8d8440dd8eff648025eb2d6ad9a219780c5",
    ),
    (
        "T12TVK.2020308.v1",
        "train",
        "validation/subsetted_512x512_HLS.S30.T12TVK.2020308.v1.4_merged.tif",
        6295168,
        "d726f700c6161238771681aad6be2a2cd140bf02ade68f93092256640c1ab32b",
        "validation/subsetted_512x512_HLS.S30.T12TVK.2020308.v1.4.mask.tif",
        525272,
        "7d815f182de219c2e88e1e9508b84df81ad928eb685104b2d9d532324d3f5c09",
    ),
    (
        "T12TWT.2020276.v1",
        "train",
        "training/subsetted_512x512_HLS.S30.T12TWT.2020276.v1.4_merged.tif",
        6295168,
        "678e4f39fac91190ce9fa1a6a13ad30b0a703e79ca202b68df8b7963792b1400",
        "training/subsetted_512x512_HLS.S30.T12TWT.2020276.v1.4.mask.tif",
        525272,
        "dc8bcc47141915172e18228af79155f9c8587b1862a40557dd24d819d754941a",
    ),
    (
        "T13RGP.2020118.v1",
        "train",
        "validation/subsetted_512x512_HLS.S30.T13RGP.2020118.v1.4_merged.tif",
        6295168,
        "94af652d046118fa24805ab01a821d0ec9152f34ec8d42951530cd5dd41b958b",
        "validation/subsetted_512x512_HLS.S30.T13RGP.2020118.v1.4.mask.tif",
        525272,
        "159b9b667f70673d9c85682a2fe016a5f4aef7cd255fbbe9a31783ac6cbe793a",
    ),
    (
        "T13SDT.2019184.v1",
        "train",
        "training/subsetted_512x512_HLS.S30.T13SDT.2019184.v1.4_merged.tif",
        6295168,
        "c0868f80ef475b61f3c87542b9173a4b80edd42dacd268fc9547cf49bc6154cf",
        "training/subsetted_512x512_HLS.S30.T13SDT.2019184.v1.4.mask.tif",
        525272,
        "5061eddc74af76014f80eb267ee0b2147b15fe42e484c658aa1aa53cdcbf117f",
    ),
    (
        "T13SFT.2019171.v1",
        "train",
        "validation/subsetted_512x512_HLS.S30.T13SFT.2019171.v1.4_merged.tif",
        6295168,
        "6ab434ca283f9749f58bb58fedda7e5664edc70629b15411192e337846874b16",
        "validation/subsetted_512x512_HLS.S30.T13SFT.2019171.v1.4.mask.tif",
        525272,
        "e2c41cfcf527154244f26ec2cf1483bb86777f172c401ebe5d3fb447d2247f4f",
    ),
    (
        "T14SLE.2019098.v1",
        "train",
        "training/subsetted_512x512_HLS.S30.T14SLE.2019098.v1.4_merged.tif",
        6295168,
        "b921e6b6e8bd2a43ed68b4f817e6dd66bf33f67363ff13d36b25649edddbd5be",
        "training/subsetted_512x512_HLS.S30.T14SLE.2019098.v1.4.mask.tif",
        525272,
        "55f34f14371eceece02f843f42e6484d12a9fe6ca84579dd865f9c7472c82b29",
    ),
    (
        "T15RWQ.2021098.v1",
        "train",
        "validation/subsetted_512x512_HLS.S30.T15RWQ.2021098.v1.4_merged.tif",
        6295168,
        "39a8a5270b1ffe8d5bba8b5cc9d0416225f4fa64e236a2bde66c29404992de67",
        "validation/subsetted_512x512_HLS.S30.T15RWQ.2021098.v1.4.mask.tif",
        525272,
        "73a1cf554c7d5539e93fe124d216001713524bb0a36b8b8c30963425f79def08",
    ),
    (
        "T15STA.2018125.v1",
        "train",
        "validation/subsetted_512x512_HLS.S30.T15STA.2018125.v1.4_merged.tif",
        6295168,
        "390a22dde0b506cbda8e96e812881701fccd6af4d16f413aaba13c7e9cfae701",
        "validation/subsetted_512x512_HLS.S30.T15STA.2018125.v1.4.mask.tif",
        525272,
        "f89131607f798f5c621968291e14b024e527a2583a5aeaaa6d733fe2b85c3205",
    ),
    (
        "T15SWV.2018099.v1",
        "train",
        "training/subsetted_512x512_HLS.S30.T15SWV.2018099.v1.4_merged.tif",
        6295168,
        "df2765625cd85db00ca90c8f3a8640a0b29a8308e82f2fca12c680c3a921fba6",
        "training/subsetted_512x512_HLS.S30.T15SWV.2018099.v1.4.mask.tif",
        525272,
        "137d6f79ab78762ea30b26503130cf7b19d265f3138c8c9145e47a4fa753c162",
    ),
    (
        "T16RFU.2019250.v1",
        "train",
        "validation/subsetted_512x512_HLS.S30.T16RFU.2019250.v1.4_merged.tif",
        6295168,
        "1ead623c4136059100bc7f1fc39daef43dfe0257e3ac038deb8e40c0152b49e6",
        "validation/subsetted_512x512_HLS.S30.T16RFU.2019250.v1.4.mask.tif",
        525272,
        "f41c8189cdc143f59fd7f4294428a538ca0cce5c5306a50e0cf13b27a7f6666c",
    ),
    (
        "T16SDD.2020093.v1",
        "train",
        "training/subsetted_512x512_HLS.S30.T16SDD.2020093.v1.4_merged.tif",
        6295168,
        "a023147cd40bf80aa15258fe704c62c51f97954021c287ba80d5ef4a9267b228",
        "training/subsetted_512x512_HLS.S30.T16SDD.2020093.v1.4.mask.tif",
        525272,
        "ff1285ca2f447dc2afdc7353222b3d7f66b90ecd0a669349501d7a7c50f0bb3a",
    ),
    (
        "T17SLT.2019112.v1",
        "train",
        "validation/subsetted_512x512_HLS.S30.T17SLT.2019112.v1.4_merged.tif",
        6295168,
        "ef628bf7fad74cad62a052ed9f208486521fa50455519671e37718e16c1ade58",
        "validation/subsetted_512x512_HLS.S30.T17SLT.2019112.v1.4.mask.tif",
        525272,
        "4ee9f13f70aabdf272b8eeaf16142fa9f350389cb1b201271a29dd6dc7fa0b58",
    ),
    (
        "T10TFM.2018110.v1",
        "validation",
        "validation/subsetted_512x512_HLS.S30.T10TFM.2018110.v1.4_merged.tif",
        6295168,
        "bfdac214cc8f46a7da6c4cd905bf38f6d33e54de9f2346df0dc3d3a289918eb4",
        "validation/subsetted_512x512_HLS.S30.T10TFM.2018110.v1.4.mask.tif",
        525272,
        "6c2b42063936b0c6d23d89b507a86bcfa0b6ca62de1ed9afce694ac513200be8",
    ),
    (
        "T11SPV.2020236.v1",
        "validation",
        "training/subsetted_512x512_HLS.S30.T11SPV.2020236.v1.4_merged.tif",
        6295168,
        "322e55a79860b9dd03d43548c805962760a073a446970b714cf7a7295644af57",
        "training/subsetted_512x512_HLS.S30.T11SPV.2020236.v1.4.mask.tif",
        525272,
        "f2cb3adc3a8cfc9f6a9ea058c2219073b897353daac32476ac5d654756b7eea7",
    ),
    (
        "T11TMF.2018222.v1",
        "validation",
        "training/subsetted_512x512_HLS.S30.T11TMF.2018222.v1.4_merged.tif",
        6295168,
        "6b8c89730c1f154b83dbcee1d6a28612a6b0885f664b35e7911daa196107c62e",
        "training/subsetted_512x512_HLS.S30.T11TMF.2018222.v1.4.mask.tif",
        525272,
        "585a5a248ed23cf6d59e81bb8405d4bd7c49a8520902beecb3b713f5d54d2020",
    ),
    (
        "T12RXV.2018217.v1",
        "validation",
        "training/subsetted_512x512_HLS.S30.T12RXV.2018217.v1.4_merged.tif",
        6295168,
        "39f7ca665d41a670a3d1544d281e563650db8829b1e995fa9319fa8e25a1edde",
        "training/subsetted_512x512_HLS.S30.T12RXV.2018217.v1.4.mask.tif",
        525272,
        "11979d66623bc1283ffc0c8cb0d7a7ad16cc467a26c4739095630120121fcc23",
    ),
    (
        "T13TBE.2018220.v1",
        "validation",
        "training/subsetted_512x512_HLS.S30.T13TBE.2018220.v1.4_merged.tif",
        6295168,
        "e85080484fa5882c0307af09faacb6c340a3f66c9c9368003ea377db5baba585",
        "training/subsetted_512x512_HLS.S30.T13TBE.2018220.v1.4.mask.tif",
        525272,
        "3aa03edb6eb327f268c8babd1cd810fed353e63925aed9b885f128a57d9f2c05",
    ),
    (
        "T13TDE.2020247.v1",
        "validation",
        "training/subsetted_512x512_HLS.S30.T13TDE.2020247.v1.4_merged.tif",
        6295168,
        "63eb63e45bb9fe41bd53a41ba078f2a9b056d34cc0fc943b5cab035d03710497",
        "training/subsetted_512x512_HLS.S30.T13TDE.2020247.v1.4.mask.tif",
        525272,
        "33ec6be65d66e2ed106b64229019f1cd307dc8fe0a4690c8047d8b39f88b1a07",
    ),
    (
        "T14SMC.2019258.v1",
        "validation",
        "training/subsetted_512x512_HLS.S30.T14SMC.2019258.v1.4_merged.tif",
        6295168,
        "c7d0b8f4f43602c89a48d0acaa1051f94cb885e1bc965268e85bd5e818bf9d17",
        "training/subsetted_512x512_HLS.S30.T14SMC.2019258.v1.4.mask.tif",
        525272,
        "fc665421e55f94af4c198a2ecd6efc5e7eb5e8df9351e286db5045739da88e40",
    ),
    (
        "T15SWA.2020109.v1",
        "validation",
        "training/subsetted_512x512_HLS.S30.T15SWA.2020109.v1.4_merged.tif",
        6295168,
        "33520267b7f304e06e12777950bb9eba25118cf905bf8782be95628c6ca4a99e",
        "training/subsetted_512x512_HLS.S30.T15SWA.2020109.v1.4.mask.tif",
        525272,
        "41bf346549fa33260c9dddb446606df5b8cc3cb7ea260eada9914e49817b6a42",
    ),
    (
        "T10SDH.2020248.v1",
        "test",
        "training/subsetted_512x512_HLS.S30.T10SDH.2020248.v1.4_merged.tif",
        6295168,
        "256825373fd01eec0da0d341d7b3f683e03d0d71e2f3e58a69f7fc3ba6ad4348",
        "training/subsetted_512x512_HLS.S30.T10SDH.2020248.v1.4.mask.tif",
        525272,
        "ad8ae249875ff8db4ee9e5cfa75b1b38538eba7bfbd4489cb1f9066fd0ef2d27",
    ),
    (
        "T10SEH.2018190.v1",
        "test",
        "validation/subsetted_512x512_HLS.S30.T10SEH.2018190.v1.4_merged.tif",
        6295168,
        "13bc592a5e569d837bd8bb3524bb0d2f28418830bcc7b0750e74033078f8b17e",
        "validation/subsetted_512x512_HLS.S30.T10SEH.2018190.v1.4.mask.tif",
        525272,
        "67e92de848f4d90730bff525d3aa51fa7c02759e494f264c0d33dcc8c4f6e66a",
    ),
    (
        "T10TFQ.2018245.v1",
        "test",
        "training/subsetted_512x512_HLS.S30.T10TFQ.2018245.v1.4_merged.tif",
        6295168,
        "c5d48a242953f23eeb06a8527204fdbd692150f4fa07cb05952251c64aa78b97",
        "training/subsetted_512x512_HLS.S30.T10TFQ.2018245.v1.4.mask.tif",
        525272,
        "cdfe51739d6bdf6ac2ac57ae307f6182b40256bfc9b8ca8a755c7a6f56c5ebdc",
    ),
    (
        "T10TFT.2018213.v1",
        "test",
        "training/subsetted_512x512_HLS.S30.T10TFT.2018213.v1.4_merged.tif",
        6295168,
        "37190ec2c2cc80eff7a7ad381a20b9a7c14518afffded118393bc56011a96368",
        "training/subsetted_512x512_HLS.S30.T10TFT.2018213.v1.4.mask.tif",
        525272,
        "ded8fbc4a7509e56dc191c54839ba4918a0b50b8e51799de37ff395f6da699f4",
    ),
    (
        "T10TGS.2018245.v1",
        "test",
        "validation/subsetted_512x512_HLS.S30.T10TGS.2018245.v1.4_merged.tif",
        6295168,
        "474fafd750cf9faeefa27f0c9c2d55dfb93c4199c87e8e249d0a7b64b8702377",
        "validation/subsetted_512x512_HLS.S30.T10TGS.2018245.v1.4.mask.tif",
        525272,
        "03a3ba34ab6c205c7fbc45cd5711194a29ffcf0aae3ec737ea9f19490601dd6c",
    ),
    (
        "T10TGT.2018188.v1",
        "test",
        "validation/subsetted_512x512_HLS.S30.T10TGT.2018188.v1.4_merged.tif",
        6295168,
        "dfc3039d03b1ae24260be63af100db7fec80d4185b82ea76914dca9234c0c5c7",
        "validation/subsetted_512x512_HLS.S30.T10TGT.2018188.v1.4.mask.tif",
        525272,
        "4cefbeaf1327e7c75ff3b040c21bc6df5a7290fc1bfa3ec1ed5b5c7196fd3e7e",
    ),
    (
        "T11TPH.2020174.v1",
        "test",
        "training/subsetted_512x512_HLS.S30.T11TPH.2020174.v1.4_merged.tif",
        6295168,
        "c0a7588d7e81c5c1d1ccafb2e6e84f1678e6d0fd927bff7a7246c4e47abe040e",
        "training/subsetted_512x512_HLS.S30.T11TPH.2020174.v1.4.mask.tif",
        525272,
        "9a5cbac806aa96f2b01a97ce3a67dfc98dc5982adf3b2489c4649c3a7549b5ed",
    ),
    (
        "T12SVD.2019183.v1",
        "test",
        "training/subsetted_512x512_HLS.S30.T12SVD.2019183.v1.4_merged.tif",
        6295168,
        "c2a899ad91ef7b81d05727f7baccedbeae93518832ae0fcab0c26d396a553cbe",
        "training/subsetted_512x512_HLS.S30.T12SVD.2019183.v1.4.mask.tif",
        525272,
        "5f2917153a3040c05ecfeda9a171bb4725d7a423471fee52c55639796aa25cd8",
    ),
    (
        "T13REQ.2018156.v1",
        "test",
        "validation/subsetted_512x512_HLS.S30.T13REQ.2018156.v1.4_merged.tif",
        6295168,
        "45583c891a46248d756d50adaf60e2feb246ecedbfcb5f69bf074c1ddf468a62",
        "validation/subsetted_512x512_HLS.S30.T13REQ.2018156.v1.4.mask.tif",
        525272,
        "9f85f2f7e50f40076caa80b0e488268688a3f397b67a671627e6a9fada97f88f",
    ),
    (
        "T13SDV.2020269.v1",
        "test",
        "training/subsetted_512x512_HLS.S30.T13SDV.2020269.v1.4_merged.tif",
        6295168,
        "98bd5b6fb0d4fd436bf0194b28e91edcb9ce0bd24a8ad20ac087a78076ec3832",
        "training/subsetted_512x512_HLS.S30.T13SDV.2020269.v1.4.mask.tif",
        525272,
        "cc10468c737d9126cc7b869e17b23ed04db2fd4705928dfc9ee34d87cb81635e",
    ),
    (
        "T13TDL.2020280.v1",
        "test",
        "validation/subsetted_512x512_HLS.S30.T13TDL.2020280.v1.4_merged.tif",
        6295168,
        "8dcfbf37dbef187f62bd89a72f9e2db5884008b9a1295cbb893a9855b83c7136",
        "validation/subsetted_512x512_HLS.S30.T13TDL.2020280.v1.4.mask.tif",
        525272,
        "625e03be07c054b1629d9183a78f25e36208e86885ace28173de83852b88badd",
    ),
    (
        "T15SXB.2020089.v1",
        "test",
        "validation/subsetted_512x512_HLS.S30.T15SXB.2020089.v1.4_merged.tif",
        6295168,
        "c0bede195f7660c6f484dab92418998920a3644c0567c3cf7e63bd2d9d5278bd",
        "validation/subsetted_512x512_HLS.S30.T15SXB.2020089.v1.4.mask.tif",
        525272,
        "e1f12ef7826981ef0fd2d03901d8bf8dab1e6f537fc9edf75413a4d03215977e",
    ),
)
SAMPLE_LABEL_SOURCE = f"{CORPUS_NAME}; {CORPUS_RELEASE}; {CORPUS_LICENSE}"


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def _sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 22), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _pinned_members() -> dict[str, tuple[int, str]]:
    out = {}
    for _name, _role, image, image_bytes, image_sha, label, label_bytes, label_sha in SAMPLE_RECORDS:
        out[image] = (image_bytes, image_sha)
        out[label] = (label_bytes, label_sha)
    return out


def _hub_download_tarball(destination: Path) -> None:
    from huggingface_hub import hf_hub_download

    hf_hub_download(DATASET_ID, TAR_NAME, repo_type="dataset", revision=DATASET_REVISION, local_dir=str(destination.parent))


def fetch_tarball(*, cache_dir: str | Path | None = None, fetcher: Any = None) -> Path:
    """The pinned dataset tarball in the cache, fetched from the Hub at the immutable revision when absent, and
    refused on a size or SHA-256 mismatch (the 2.6 GB file is hashed once per call)."""
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    cache.mkdir(parents=True, exist_ok=True)
    local = cache / TAR_NAME
    if not local.is_file() or local.stat().st_size != TAR_BYTES:
        if fetcher is not None:
            local.write_bytes(fetcher(CORPUS_BASE_URL + TAR_NAME))
        else:
            _hub_download_tarball(local)
    size = local.stat().st_size
    digest = _sha256_file(local)
    if size != TAR_BYTES or digest != TAR_SHA256:
        raise ValueError(f"{TAR_NAME}: {size} bytes with sha256 {digest[:16]}…, pinned {TAR_BYTES} / {TAR_SHA256[:16]}…")
    return local


def extract_pinned_members(tar_path: str | Path, *, cache_dir: str | Path | None = None) -> dict[str, bytes]:
    """Stream through the tarball once and copy out exactly the pinned members (no `extractall`, no paths from
    the archive: each is written under its base name in `cache_dir/chips/`), refusing a size or digest mismatch."""
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    chips = cache / "chips"
    chips.mkdir(parents=True, exist_ok=True)
    wanted = _pinned_members()
    out: dict[str, bytes] = {}
    with tarfile.open(tar_path, "r:gz") as archive:
        for member in archive:
            if member.name not in wanted or not member.isfile():
                continue
            size, sha = wanted[member.name]
            handle = archive.extractfile(member)
            data = handle.read() if handle is not None else b""
            if len(data) != size or _sha256_bytes(data) != sha:
                raise ValueError(
                    f"{member.name}: {len(data)} bytes with sha256 {_sha256_bytes(data)[:16]}…, pinned {size} / {sha[:16]}…"
                )
            (chips / Path(member.name).name).write_bytes(data)
            out[member.name] = data
    missing = sorted(set(wanted) - set(out))
    if missing:
        raise ValueError(f"tarball does not contain {len(missing)} pinned members, e.g. {missing[:3]}")
    return out


def fetch_corpus(*, cache_dir: str | Path | None = None, fetcher: Any = None) -> dict[str, dict[str, bytes]]:
    """Every pinned scene's image and mask bytes, keyed by scene key: from the extracted cache when every file is
    present with its pinned digest, otherwise from the (verified) tarball."""
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    chips = cache / "chips"
    wanted = _pinned_members()
    cached: dict[str, bytes] = {}
    for member, (size, sha) in wanted.items():
        local = chips / Path(member).name
        if local.is_file() and local.stat().st_size == size:
            data = local.read_bytes()
            if _sha256_bytes(data) == sha:
                cached[member] = data
    if len(cached) != len(wanted):
        cached = extract_pinned_members(fetch_tarball(cache_dir=cache, fetcher=fetcher), cache_dir=cache)
    out = {}
    for name, _role, image, *_rest in SAMPLE_RECORDS:
        label = _rest[2]
        out[name] = {"image": cached[image], "label": cached[label]}
    return out


def read_corpus(files: Mapping[str, Mapping[str, bytes]]) -> dict[str, list[dict[str, Any]]]:
    """Decode the verified bytes into `{id, image, label}` records grouped by role (train / validation / test)."""
    import tempfile

    splits: dict[str, list[dict[str, Any]]] = {role: [] for role in ROLES}
    for name, role, image_member, *_rest in SAMPLE_RECORDS:
        if name not in files:
            raise ValueError(f"corpus is missing {name}")
        with tempfile.TemporaryDirectory() as tmp:
            image_path = Path(tmp) / "image.tif"
            label_path = Path(tmp) / "label.tif"
            image_path.write_bytes(files[name]["image"])
            label_path.write_bytes(files[name]["label"])
            image = read_chip(image_path)
            label = read_mask(label_path)
        raw = {
            "id": f"{role}-{len(splits[role]):03d}",
            "source_id": name,
            "region": name.split(".")[0],  # the HLS tile id (UTM zone + grid square)
            "split": role,
            "image": image,
            "label": label,
            "source": f"{CORPUS_BASE_URL}{TAR_NAME}#{image_member}",
        }
        splits[role].append(check_record(raw))  # no-data replaced, range checked, label checked
    return splits


def fetch_sample_dataset(*, cache_dir: str | Path | None = None, fetcher: Any = None) -> dict[str, list[dict[str, Any]]]:
    """The tutorial splits from the pinned corpus (roles from the model repository's splits)."""
    return read_corpus(fetch_corpus(cache_dir=cache_dir, fetcher=fetcher))


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no chip (by pixel digest) appears in two splits (leakage check)."""
    seen: dict[str, str] = {}
    for name, records in splits.items():
        for record in records:
            key = chip_digest(record)
            if key in seen and seen[key] != name:
                raise ValueError(f"chip {record['id']!r} appears in both {seen[key]} and {name}")
            seen[key] = name
    return {name: len(records) for name, records in splits.items()}


def split_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    val_fraction: float = 0.2,
    test_fraction: float = 0.25,
    seed: int = 0,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded shuffle of a BYOD dataset into train / validation / test after de-duplicating chips. Chips from one
    fire or one tile are near-duplicates; group them yourself (one fire per split) when that matters."""
    import random

    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records)["records"]
    seen: set[str] = set()
    unique = []
    for record in checked:
        key = chip_digest(record)
        if key not in seen:
            seen.add(key)
            unique.append(record)
    rng = random.Random(seed)
    rng.shuffle(unique)
    n_test = max(1, round(len(unique) * test_fraction))
    n_val = round(len(unique) * val_fraction)
    splits = {"test": unique[:n_test], "validation": unique[n_test : n_test + n_val], "train": unique[n_test + n_val :]}
    if len(splits["train"]) < MIN_RECORDS:
        raise ValueError(f"split leaves {len(splits['train'])} training chips; at least {MIN_RECORDS} are required")
    return splits


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Read `{id, image, label}` records from a directory or a zip holding `pairs.csv` (columns `id`, `image`,
    `label`) beside six-band 512 × 512 GeoTIFF chips and single-band label rasters; files are decoded from bytes,
    never extracted to disk."""
    import tempfile

    source = Path(path)
    if source.is_dir():
        table = (source / "pairs.csv").read_text(encoding="utf-8")
        loader = lambda name: (source / name).read_bytes()  # noqa: E731
    elif source.is_file() and source.suffix.lower() == ".zip":
        archive = zipfile.ZipFile(source)
        members = {Path(n).name: n for n in archive.namelist()}
        if "pairs.csv" not in members:
            raise ValueError("BYOD zip must contain pairs.csv")
        table = archive.read(members["pairs.csv"]).decode("utf-8")
        loader = lambda name: archive.read(members[name])  # noqa: E731
    else:
        raise ValueError("BYOD datasets must be a directory or a .zip holding pairs.csv and the GeoTIFF files")
    rows = list(csv.DictReader(io.StringIO(table)))
    missing = {"id", "image", "label"} - set(rows[0].keys() if rows else set())
    if missing:
        raise ValueError(f"pairs.csv is missing columns {sorted(missing)}")
    out = []
    with tempfile.TemporaryDirectory() as tmp:
        for row in rows:
            image_path = Path(tmp) / "image.tif"
            image_path.write_bytes(loader(row["image"]))
            record: dict[str, Any] = {"id": row["id"], "image": read_chip(image_path)}
            if row.get("label"):
                label_path = Path(tmp) / "label.tif"
                label_path.write_bytes(loader(row["label"]))
                record["label"] = read_mask(label_path)
            out.append(record)
    return out


def write_sample_pair(record: Mapping[str, Any], image_path: str | Path, label_path: str | Path) -> dict[str, str]:
    """Write one record as a six-band float32 TIFF and a single-band int16 TIFF (the BYOD shape, without
    georeferencing) and return both paths."""
    import numpy as np
    import tifffile

    image_out, label_out = Path(image_path), Path(label_path)
    image_out.parent.mkdir(parents=True, exist_ok=True)
    tifffile.imwrite(image_out, np.asarray(record["image"], dtype=np.float32), photometric="minisblack", planarconfig="separate")
    tifffile.imwrite(label_out, np.asarray(record["label"], dtype=np.int16), photometric="minisblack")
    return {"image": str(image_out), "label": str(label_out)}


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """Write the pairs table of a split (id, image, label, provenance) in the shape BYOD expects."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=["id", "image", "label", "region", "source"])
        writer.writeheader()
        for record in records:
            writer.writerow(
                {
                    "id": record["id"],
                    "image": f"{record.get('source_id', record['id'])}_merged.tif",
                    "label": f"{record.get('source_id', record['id'])}.mask.tif",
                    "region": record.get("region", ""),
                    "source": record.get("source", ""),
                }
            )
    return out


def dataset_manifest(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Validate every split and summarise the dataset (counts, class balance, digests) for provenance exports."""
    summary: dict[str, Any] = {"model_id": MODEL_ID, "image_size": IMAGE_SIZE, "splits": {}}
    for name, records in splits.items():
        report = validate_dataset(records, min_records=1)
        summary["splits"][name] = {
            "n_records": report["n_records"],
            "class_pixel_fraction": report["class_pixel_fraction"],
            "ignored_pixels": report["ignored_pixels"],
            "regions": sorted({str(r.get("region", "")) for r in records if r.get("region")}),
            "digest": report["digest"],
        }
    summary["disjoint"] = check_split_disjoint(splits)
    digests = json.dumps({k: v["digest"] for k, v in summary["splits"].items()}, sort_keys=True)
    summary["digest"] = hashlib.sha256(digests.encode("utf-8")).hexdigest()
    return summary

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `10`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `a3f2c410e45b…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `PrithviBurnScarPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, device=('cuda' if torch.cuda.is_available() else 'cpu'), report=print)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "prithvi-eo-2.0-300m-burnscars",
  "modelId": "ibm-nasa-geospatial/Prithvi-EO-2.0-300M-BurnScars",
  "revision": "a3f2c410e45b8ac7417976614528a872f024d831",
  "files": [
    {
      "path": "README.md",
      "bytes": 3717,
      "sha256": "714e8139bcdad0b9ea32628f51460fbcaada07085edd9c026a9daa5738fb7532"
    },
    {
      "path": "config.json",
      "bytes": 3228,
      "sha256": "a7e2ca5f4555820bfc9ec92a11142a4fefdade7033aa6653965c179a384b115e"
    },
    {
      "path": "burn_scars_config.yaml",
      "bytes": 2504,
      "sha256": "9b40e6d8c6369b863d29fb1d69b6185aa366f8ee4c2de955bbad28d3ee32f466"
    },
    {
      "path": "Prithvi_EO_V2_300M_BurnScars.pt",
      "bytes": 1297798380,
      "sha256": "0c5f9334be9a75c9006387ab8f3dc05a55ea7fb5ef7956717316be57c62954d3"
    },
    {
      "path": "splits/train.txt",
      "bytes": 9432,
      "sha256": "f1dca5c488c7a7451f3aa82f20677c18b3b22a2d96b59d68215494e3dba6f393"
    },
    {
      "path": "splits/val.txt",
      "bytes": 2880,
      "sha256": "50a98be7974a2fec73af62b5f54063d938bcfe93cd9802b90eb33eb78f4d7f3e"
    },
    {
      "path": "splits/test.txt",
      "bytes": 2160,
      "sha256": "c61ddcb5a434da3f5e63edd7c87549af104c6cc69345876a8ed770844bab1788"
    },
    {
      "path": "examples/subsetted_512x512_HLS.S30.T10SEH.2018190.v1.4_merged.tif",
      "bytes": 6295168,
      "sha256": "13bc592a5e569d837bd8bb3524bb0d2f28418830bcc7b0750e74033078f8b17e"
    },
    {
      "path": "examples/subsetted_512x512_HLS.S30.T10SFF.2018190.v1.4_merged.tif",
      "bytes": 6295168,
      "sha256": "b491445bcca5d23a534765ac9f8b24f4cb0c9a75a7254c969456d65a982207a5"
    },
    {
      "path": "examples/subsetted_512x512_HLS.S30.T10SGF.2020217.v1.4_merged.tif",
      "bytes": 6295168,
      "sha256": "ecaa478fdb21ed437ea03436da87ed5efbd3d980e4da23cbee05171212c40378"
    }
  ],
  "totalBytes": 1316707805
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = PrithviBurnScarPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, device=('cuda' if torch.cuda.is_available() else 'cpu'), report=print)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Sample scenes, validation and roles

The default dataset is 44 labelled HLS scenes of the HLS Burn Scars dataset — 24 from the training split, 8 from the validation split and 12 from the test split that the upstream authors publish beside the checkpoint, drawn with a fixed seed from the scenes whose mask is at least 60 % valid and 3 % burn scar. `fetch_corpus` downloads the dataset tarball from the Hub at its immutable revision, refuses it on any size or SHA-256 mismatch, streams through it once and copies out exactly the 88 pinned members — each refused on its own size or digest mismatch and written under its base name, never at a path taken from the archive — then reads the six-band float32 scenes (already surface reflectance in [0, 1]) and the masks, which keep −1 for no data. `dataset_manifest` validates every split, checks that no scene appears twice and records a digest.

Look for: 24 / 8 / 12 scenes with burn fractions around 0.12..0.21, tile ids (UTM zone and grid square) per split, a written sample pair (`outputs/prithvi_burnscar_segmentation_sample_chip.tif` + `_sample_label.tif`, the BYOD shape), and three refusal probes — a five-band scene, a mask with an unknown class, a scene with reflectance far outside range — each rejected before the model runs. The tarball takes about a minute to fetch and a minute to stream.

In [ ]:
import json
import os
from pathlib import Path

import numpy as np

USE_BYOD = False  # @param {type:"boolean"}

os.makedirs('outputs', exist_ok=True)
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_path = Path('work') / file_name
    byod_path.parent.mkdir(parents=True, exist_ok=True)
    byod_path.write_bytes(payload)
    splits = split_dataset(load_byod_dataset(byod_path), seed=0)
    data_source = 'BYOD (' + file_name + ')'
else:
    splits = fetch_sample_dataset(cache_dir='weights/hls-burn-scars')
    data_source = SAMPLE_LABEL_SOURCE
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']

dataset_report = dataset_manifest({'train': train_records, 'validation': val_records, 'test': test_records})
print({'data_source': data_source, 'splits': {k: v['n_records'] for k, v in dataset_report['splits'].items()}, 'disjoint': dataset_report['disjoint'], 'digest': dataset_report['digest'][:16] + '...'})
for name, part in dataset_report['splits'].items():
    print({name: {'burn_fraction': part['class_pixel_fraction']['burn scar'], 'ignored_pixels': part['ignored_pixels'], 'tiles': part['regions']}})
print({'first_test_scene': validate_inputs(test_records[0])})
sample_pair = write_sample_pair(test_records[0], 'outputs/prithvi_burnscar_segmentation_sample_chip.tif', 'outputs/prithvi_burnscar_segmentation_sample_label.tif')
print({'sample_pair': sample_pair, 'pairs_csv': str(write_dataset_csv(test_records, 'outputs/prithvi_burnscar_segmentation_sample_pairs.csv'))})

print({'validation': INPUT_SCHEMA['validation']})
probes = {
    'five-band scene': [{**test_records[0], 'image': test_records[0]['image'][:5]}, *test_records[1:4]],
    'unknown label class': [{**test_records[0], 'label': np.where(test_records[0]['label'] == 1, 7, test_records[0]['label'])}, *test_records[1:4]],
    'reflectance out of range': [{**test_records[0], 'image': test_records[0]['image'] * 50000.0}, *test_records[1:4]],
}
for name, records in probes.items():
    try:
        validate_dataset(records)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

## 5. The frozen model against the not-burned baseline

`pipe.predict` standardises each scene with the band statistics of the upstream training configuration, runs the encoder, neck, decoder and head in float16 autocast, and returns the argmax mask, the softmax scores (the model's outputs, not calibrated probabilities) and the burn fraction per scene. `pipe.evaluate` pools the labelled pixels of every held-out scene into one confusion matrix (−1 pixels excluded) and reports the per-class IoU, mean IoU, accuracy, and the burn-scar class's precision, recall and F1; the **not-burned baseline** — every pixel predicted as unburned — is scored on the same pixels, so its accuracy is exactly the unburned fraction and its burn IoU is 0.

Look for: a burn-scar IoU above 0.9 on the test scenes (in the build record 0.925 with F1 0.961 — this fine-tune is strong on its own test split, from which these scenes were drawn) and a lower validation IoU (about 0.76: a few validation scenes are hard). These are sample-sanity numbers on 12 and 8 scenes, not the benchmark.

In [ ]:
import time

t0 = time.perf_counter()
frozen_test = pipe.evaluate(test_records)
frozen_val = pipe.evaluate(val_records)
print({'seconds': round(time.perf_counter() - t0, 1), 'metric': frozen_test['metric']})
print({'baseline_not_burned_test': {k: frozen_test['baseline_not_burned'][k] for k in ('iou', 'accuracy', 'f1')}})
print({'frozen_test': {k: frozen_test['model'][k] for k in ('iou', 'mean_iou', 'accuracy', 'precision', 'recall', 'f1')}})
print({'frozen_validation': {k: frozen_val['model'][k] for k in ('iou', 'f1')}})
frozen_predictions = pipe.predict(test_records)
for record, pred in list(zip(test_records, frozen_predictions['predictions']))[:6]:
    labelled = record['label'] >= 0
    print({'scene': record['source_id'], 'burn_label': round(float((record['label'] == 1).sum() / labelled.sum()), 3), 'burn_predicted': pred['class_fraction']['burn scar'], 'ignored': int((~labelled).sum())})
print({'decision_rule': frozen_predictions['decision_rule'], 'scores_shape': frozen_predictions['predictions'][0]['scores'].shape})

## 6. Bounded fine-tuning of the neck, decoder and head

`pipe.adapt` trains the 34 tensors of the pyramid neck, the U-Net decoder and the head (20.3 M parameters — 6.3 % of the model) and nothing else: the ViT-L encoder is frozen (no gradient is stored for it), and every BatchNorm layer keeps its running statistics, because batches of two scenes would corrupt them. Each step takes two scenes with a seeded horizontal or vertical flip, computes the cross-entropy over the labelled pixels (−1 ignored) and takes an AdamW step at a small fixed learning rate with gradient-norm clipping and float16 loss scaling. Epoch 0 records the frozen model's validation loss and metrics; the epoch with the lowest validation loss is kept — which can be epoch 0, since the packaged model already trained on this dataset.

Watch the validation loss: in the build record it dipped at epoch 1 and drifted up afterwards — the sign that a small learning rate and validation selection are doing their job on a model that has little left to learn from 24 scenes of a dataset it trained on. Four epochs (48 steps) take under a minute on a T4. `TRAINABLE = 'decoder+last_block'` also unfreezes the last encoder block (32.9 M parameters).

In [ ]:
EPOCHS = 4  # @param {type:"integer"}
LEARNING_RATE = 1e-5  # @param {type:"number"}
BATCH_SIZE = 2  # @param {type:"integer"}
TRAINABLE = 'decoder'  # @param ["decoder", "decoder+last_block"]

def report(entry):
    row = {'epoch': entry['epoch'], 'train_loss': None if entry['train_loss'] is None else round(entry['train_loss'], 4), 'val_loss': round(entry['val_loss'], 4)}
    if 'val' in entry:
        row['val_burn_iou'] = entry['val']['iou']['burn scar']
        row['val_f1'] = entry['val']['f1']
    if 'note' in entry:
        row['note'] = entry['note']
    print(row)

t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, trainable=TRAINABLE, progress=report)
adapt_seconds = round(time.perf_counter() - t0, 1)
print({'trainable_parameters': adapt_result['n_trainable'], 'total_parameters': adapt_result['n_total'], 'steps': adapt_result['n_steps'], 'best_epoch': adapt_result['best_epoch'], 'precision': adapt_result['precision'], 'batchnorm': adapt_result['batchnorm'], 'seconds': adapt_seconds})

## 7. Held-out evaluation: the paired comparison

The test scenes were never used for training or epoch selection (they come from the model repository's test split). The adapted model is scored exactly as the frozen model was in Section 5, and the table puts the baseline, the frozen and the adapted numbers side by side. The cell asserts what the procedure guarantees — the kept epoch's validation loss is no higher than the frozen model's, and re-scoring the validation scenes reproduces the kept epoch's burn-scar IoU within 0.01 (float16 kernels are not bit-reproducible across batch sizes) — and prints the test numbers without asserting a direction: on this sample the burn-scar IoU moved from 0.925 to 0.926 in the build record, a sample-sanity observation on 12 scenes with no dispersion estimate, not a quality claim. With your own scenes from another region or year, the gap between frozen and adapted is the number to watch.

In [ ]:
adapted_test = pipe.evaluate(test_records)
adapted_val = pipe.evaluate(val_records)
comparison = {}
for key in ('mean_iou', 'accuracy', 'precision', 'recall', 'f1'):
    comparison[key] = {'baseline_not_burned': frozen_test['baseline_not_burned'][key], 'frozen': frozen_test['model'][key], 'adapted': adapted_test['model'][key]}
comparison['burn_iou'] = {'baseline_not_burned': frozen_test['baseline_not_burned']['iou']['burn scar'], 'frozen': frozen_test['model']['iou']['burn scar'], 'adapted': adapted_test['model']['iou']['burn scar']}
for key, row in comparison.items():
    print({key: row})
print({'validation_burn_iou': {'frozen': frozen_val['model']['iou']['burn scar'], 'adapted': adapted_val['model']['iou']['burn scar']}, 'validation_loss': {'frozen': adapt_result['history'][0]['val_loss'], 'kept_epoch': adapt_result['history'][adapt_result['best_epoch']]['val_loss']}})
evaluation_report = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': MODEL_KEY},
    'data_source': data_source,
    'dataset': dataset_report,
    'frozen': {'test': frozen_test, 'validation': frozen_val},
    'adapted': {'test': adapted_test, 'validation': adapted_val},
    'comparison': comparison,
    'adaptation': {k: v for k, v in adapt_result.items() if k not in ('history', 'trainable_names')},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/prithvi_burnscar_segmentation_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report, f, indent=2)
assert adapt_result['history'][adapt_result['best_epoch']]['val_loss'] <= adapt_result['history'][0]['val_loss']
assert abs(adapted_val['model']['iou'][CLASS_NAMES[1]] - adapt_result['history'][adapt_result['best_epoch']]['val']['iou'][CLASS_NAMES[1]]) < 1e-2
print({'report': 'outputs/prithvi_burnscar_segmentation_evaluation_report.json'})

## 8. New scenes, artifact export and fresh reload

The adapted model segments the three example scenes that ship with the upstream repository (three HLS tiles over California, 2018 and 2020), which carry no labels here: the predicted burn fraction per scene and a written mask are a sanity check, not an evaluation.

`pipe.save_artifact` writes the trained tensors (about 81 MB) as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digest of the converted base file, the adaptation scope, the tensor names, the file size and SHA-256, the training configuration and the epoch history (OUT8). `PrithviBurnScarPipeline.from_artifact` re-verifies the base file, checks the artifact manifest, scope and digest **before** deserialising, rebuilds the model and overlays the tensors — a fresh object from files, not the in-memory model (VER2). The cell asserts the same held-out burn-scar IoU within 0.001 and score maps within 0.01 (VER4: float16 tolerances; on one device they are usually identical).

In [ ]:
import platform
import shutil

import tifffile

example_dir = WEIGHTS_DIR / 'examples'
new_records = [{'id': path.stem.replace('subsetted_512x512_HLS.S30.', ''), 'image': read_chip(path), 'source': str(path.name)} for path in sorted(example_dir.glob('*.tif'))]
new_predictions = pipe.predict(new_records)
for record, pred in zip(new_records, new_predictions['predictions']):
    tifffile.imwrite(f'outputs/prithvi_burnscar_segmentation_mask_' + record['id'] + '.tif', pred['mask'])
    print({'scene': record['id'], 'burn_fraction': pred['class_fraction']['burn scar'], 'note': 'sanity check, no label'})
with open('outputs/prithvi_burnscar_segmentation_predictions.json', 'w', encoding='utf-8') as f:
    json.dump({'model': new_predictions['model'], 'classes': new_predictions['classes'], 'decision_rule': new_predictions['decision_rule'], 'predictions': [{'id': p['id'], 'class_fraction': p['class_fraction']} for p in new_predictions['predictions']]}, f, indent=2)

artifact_dir = Path('outputs/prithvi_burnscar_segmentation_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'prithvi_burnscar_segmentation', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'trainable': artifact_manifest['adapter']['trainable'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...'})

reloaded = PrithviBurnScarPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, device=pipe.device)
reloaded_test = reloaded.evaluate(test_records)
before = pipe.predict(test_records[:2])['predictions']
after = reloaded.predict(test_records[:2])['predictions']
parity = {'positive_iou_diff': round(abs(reloaded_test['model']['iou'][CLASS_NAMES[1]] - adapted_test['model']['iou'][CLASS_NAMES[1]]), 6), 'metrics_identical': reloaded_test['model'] == adapted_test['model'], 'max_abs_score_diff': max(float(np.abs(a['scores'] - b['scores']).max()) for a, b in zip(before, after))}
print({'reload_parity': parity, 'reloaded_best_epoch': reloaded.adapter['best_epoch']})
assert parity['positive_iou_diff'] < 1e-3 and parity['max_abs_score_diff'] < 1e-2

result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model': {**evaluation_report['model'], 'model_license': MODEL_LICENSE, 'device': pipe.device, 'source': pipe.source},
    'provenance': {
        'source_asset': [e for e in MANIFEST['files'] if e['path'] == SOURCE_CKPT_NAME],
        'pickle_audit_sha256': PICKLE_AUDIT_SHA256,
        'converted': verify_converted(WEIGHTS_DIR)['files'],
        'pickle_unpickled_once_for_conversion': True,
        'served_from_pickle': False,
        'remote_code_executed': False,
        'data_tarball': {'name': TAR_NAME, 'sha256': TAR_SHA256, 'pinned_members': 2 * len(SAMPLE_RECORDS)},
        'data_base_url': CORPUS_BASE_URL,
        'data_license': CORPUS_LICENSE,
    },
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'timm': timm.__version__, 'lightning': lightning.__version__, 'tifffile': tifffile.__version__, 'terratorch': importlib.metadata.version('terratorch')},
    'data_source': data_source,
    'comparison': comparison,
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes']},
    'reload_parity': parity,
}
with open('outputs/prithvi_burnscar_segmentation_result.json', 'w', encoding='utf-8') as f:
    json.dump(result_payload, f, indent=2)

print('outputs/:')
for path in sorted(Path('outputs').rglob('*')):
    if path.is_file():
        print(f'  - {path.as_posix()} ({path.stat().st_size / 1024:.1f} KB)')

## Interpretation and limits

On 12 held-out scenes from the model repository's test split the packaged burn-scar model finds burn scars with an IoU above 0.9, against a not-burned baseline that scores 0; a bounded fine-tuning of its neck, decoder and head on 24 scenes, selected by validation loss with the frozen model as a candidate, leaves those numbers where they were. That is the claim: the adaptation contract runs end to end on real labelled multispectral scenes drawn from a digest-verified tarball, the pickle is audited and converted rather than served, and the artifact that carries the change is 81 MB and reloads with the same outputs. It is not a claim that this sample improves the model — the model already trained on this dataset — nor that 12 scenes measure its skill.

The numbers are sample-sanity evidence: one seeded run, 12 test scenes from a handful of HLS tiles, no dispersion estimate, pixel-pooled metrics that let large burns dominate, and labels derived from a burned-area product with their own uncertainty at scar edges and under smoke. Nothing here measures the model outside the contiguous United States, outside 2018–2021, on scenes larger than a chip, or on burn severity.

Three things to carry to real data. **The six bands and their scaling are the contract:** blue, green, red, narrow NIR, SWIR 1, SWIR 2 in that order, surface reflectance in [0, 1]; a different band order or an uncorrected product is segmented without complaint and silently wrong. **Split by fire or tile, not by scene:** neighbouring scenes of one fire are near-duplicates, and a random split makes memorisation look like skill. **Read the baseline first:** on a scene with 5 % burn the not-burned baseline is 95 % accurate; only the burn-scar IoU, precision and recall say whether the model did anything.

Successful execution proves that the recorded repository revision's pipeline modules, carried in this standalone notebook, can acquire and digest-verify a pickled upstream checkpoint, audit and convert it into safetensors without executing anything outside the audited allow-list, rebuild the model from the installed package, fetch a digest-pinned tarball and extract exactly the pinned labelled scenes, execute bounded fine-tuning, evaluate against a baseline and the frozen model on held-out scenes, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, production fitness, or burn-mapping skill beyond the checks shown.

**Optional experiments (they do not affect the default path):** set `TRAINABLE = 'decoder+last_block'`; raise `EPOCHS` and watch the validation loss drift; try `LEARNING_RATE = 1e-4` to see the frozen model win every epoch; or bring your own labelled scenes through BYOD and read the baseline before the adapted number.

## References

- Repository README: https://github.com/kurtvalcorza/prithvi-burnscar-segmentation-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/prithvi-burnscar-segmentation-pipeline/blob/main/MODEL_CARD.md
- Weights and conversion notes: https://github.com/kurtvalcorza/prithvi-burnscar-segmentation-pipeline/blob/main/docs/WEIGHTS.md
- Hugging Face model repository: https://huggingface.co/ibm-nasa-geospatial/Prithvi-EO-2.0-300M-BurnScars (revision `a3f2c410e45b8ac7417976614528a872f024d831`)
- HLS Burn Scars dataset: https://huggingface.co/datasets/ibm-nasa-geospatial/hls_burn_scars (CC BY 4.0)
- Szwarcman, D., Roy, S., Fraccaro, P., et al. (2024). Prithvi-EO-2.0: A versatile multi-temporal foundation model for Earth observation applications. arXiv:2412.02732: https://arxiv.org/abs/2412.02732
- TerraTorch: https://github.com/IBM/terratorch
- DIMER Notebook Specification 2.0 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)